In [2]:
from __future__ import annotations

import sys
import subprocess
import importlib.util

PACKAGE_IMPORT_MAP = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "tqsdk": "tqsdk",
}

missing_packages = [
    package
    for package, module_name in PACKAGE_IMPORT_MAP.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("Required packages already available.")

import hashlib
import math
import os
import re
import warnings
from contextlib import closing
from dataclasses import dataclass
from datetime import date, timedelta
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    display
except NameError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=RuntimeWarning)
try:
    warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)
except Exception:
    pass

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


Required packages already available.


In [6]:
PROJECT_DIR = Path("main_continuous_daily_trend_momentum_reversal_research")
RAW_MAIN_DIR = PROJECT_DIR / "raw" / "tq_main_continuous_daily"
PROCESSED_DIR = PROJECT_DIR / "processed"
RESULT_DIR = PROJECT_DIR / "results"
FIG_DIR = PROJECT_DIR / "figures"

for p in [RAW_MAIN_DIR, PROCESSED_DIR, RESULT_DIR, FIG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# 运行开关
RUN_TQ_DOWNLOAD = False
AUTO_DOWNLOAD_IF_MISSING = True
OVERWRITE_RAW_CSV = False
RUN_PLOTS = False

# 日线参数
DAILY_KLINE_DATA_LENGTH = 8000
DAILY_KLINE_DUR_SEC = 24 * 60 * 60

# 研究区间
START_DATE = date(1990, 1, 1)
# Frozen research snapshot date.
# For rolling daily diagnostics, replace with: END_DATE = date.today()
END_DATE = date(2026, 6, 30)
LOCAL_TIMEZONE = "Asia/Shanghai"

# 品种起始过滤日期
PRODUCT_START_DATE = {
    "M": date(2000, 1, 1),
    "SP": date(2018, 11, 27),
    "FG": date(2012, 12, 3),
    "SA": date(2019, 12, 6),

    "AG": date(2012, 5, 10),
    "V": date(2009, 5, 25),
    "LH": date(2021, 1, 8),
    "EB": date(2019, 9, 26),
    "MA": date(2011, 10, 28),
    "CF": date(2004, 6, 1),
    "SR": date(2006, 1, 6),
}

MIN_OHLCV_VOLUME = 0
MIN_SIGMA = 1e-6
VOL_BASE_LOOKBACK = 20
EWMA_LAMBDA = 0.94
ATR_STOP_LOOKBACK = 20

H_RET_LIST = [1, 2, 3, 5, 10, 20, 40, 60]
J_LIST = [2, 3, 5, 10, 20, 40, 60]
K_LIST = [1, 2, 3, 5, 10, 20]

SHOCK_Z_THRESHOLD = 2.0

TSREV_Z_THRESHOLD = 1.0
RAREV_Z_THRESHOLD = 1.0
FAST_REV_Z = 2.0
REV_SIGNAL_MIN_ABS = 1e-12

# 反转确立口径：raw reversal signal 必须在同一方向上连续出现 N 天，才进入 validated reversal。
REV_ESTABLISH_CONFIRM_DAYS = 2

MIN_MODEL_N = 60
MIN_CONFIRMED_N = 120
MIN_T_CAND = 1.0
MIN_T_CONF = 2.0
MIN_HIT_RATE = 0.50
BOOTSTRAP_BLOCK = 5
BOOTSTRAP_N = 400
RANDOM_SEED = 20260630

rng = np.random.default_rng(RANDOM_SEED)


In [7]:
@dataclass(frozen=True)
class ProductSpec:
    commodity: str
    name: str
    main_symbol: str


PRODUCTS = {
    "M": ProductSpec("M", "豆粕", "KQ.m@DCE.m"),
    "SP": ProductSpec("SP", "纸浆", "KQ.m@SHFE.sp"),
    "FG": ProductSpec("FG", "玻璃", "KQ.m@CZCE.FG"),
    "SA": ProductSpec("SA", "纯碱", "KQ.m@CZCE.SA"),

    "AG": ProductSpec("AG", "沪银", "KQ.m@SHFE.ag"),
    "V": ProductSpec("V", "PVC", "KQ.m@DCE.v"),
    "LH": ProductSpec("LH", "生猪", "KQ.m@DCE.lh"),
    "EB": ProductSpec("EB", "苯乙烯", "KQ.m@DCE.eb"),
    "MA": ProductSpec("MA", "甲醇", "KQ.m@CZCE.MA"),
    "CF": ProductSpec("CF", "棉花", "KQ.m@CZCE.CF"),
    "SR": ProductSpec("SR", "白糖", "KQ.m@CZCE.SR"),
}

# 天勤账号：仅从环境变量读取，避免在 notebook 中保存明文账号密码
TQ_USERNAME = os.getenv("TQ_USERNAME", "").strip()
TQ_PASSWORD = os.getenv("TQ_PASSWORD", "").strip()

print("研究品种：")
for k, v in PRODUCTS.items():
    print(f"  {k}: {v.main_symbol}, name={v.name}")

print("\n是否在 Cell 04 主动下载：", RUN_TQ_DOWNLOAD)
print("缺失数据时是否自动下载：", AUTO_DOWNLOAD_IF_MISSING)
print("当前工作目录：", Path.cwd().resolve())
print("主力连续合约 CSV 目录：", RAW_MAIN_DIR.resolve())


研究品种：
  M: KQ.m@DCE.m, name=豆粕
  SP: KQ.m@SHFE.sp, name=纸浆
  FG: KQ.m@CZCE.FG, name=玻璃
  SA: KQ.m@CZCE.SA, name=纯碱
  AG: KQ.m@SHFE.ag, name=沪银
  V: KQ.m@DCE.v, name=PVC
  LH: KQ.m@DCE.lh, name=生猪
  EB: KQ.m@DCE.eb, name=苯乙烯
  MA: KQ.m@CZCE.MA, name=甲醇
  CF: KQ.m@CZCE.CF, name=棉花
  SR: KQ.m@CZCE.SR, name=白糖

是否在 Cell 04 主动下载： False
缺失数据时是否自动下载： True
当前工作目录： /home/zilinm2
主力连续合约 CSV 目录： /home/zilinm2/main_continuous_daily_trend_momentum_reversal_research/raw/tq_main_continuous_daily


In [9]:
def safe_filename(symbol: str) -> str:
    return re.sub(r"[^A-Za-z0-9_]+", "_", symbol)


def main_csv_path(commodity: str, symbol: str) -> Path:
    return RAW_MAIN_DIR / f"{commodity}_{safe_filename(symbol)}_daily.csv"


def has_tq_credentials() -> bool:
    return bool(str(TQ_USERNAME).strip()) and bool(str(TQ_PASSWORD).strip())


def create_tq_api():
    if not has_tq_credentials():
        raise RuntimeError(
            "缺少天勤账号。请设置环境变量 TQ_USERNAME / TQ_PASSWORD，"
            "或关闭 RUN_TQ_DOWNLOAD / AUTO_DOWNLOAD_IF_MISSING 并提供本地 CSV。"
        )

    try:
        from tqsdk import TqApi, TqAuth
    except ImportError as e:
        raise RuntimeError(
            "需要下载天勤数据，但当前环境未安装 tqsdk。"
            "请先安装 tqsdk，或关闭 RUN_TQ_DOWNLOAD / AUTO_DOWNLOAD_IF_MISSING 并提供本地 CSV。"
        ) from e

    return TqApi(auth=TqAuth(TQ_USERNAME, TQ_PASSWORD))


def parse_numeric_datetime_series(x: pd.Series) -> pd.Series:
    """
    解析 TqSdk K 线 datetime。
    关键点：get_kline_serial 前部可能有 datetime=0 的占位行，不能参与时间单位判断。
    """
    x_num = pd.to_numeric(x, errors="coerce")
    positive = x_num[x_num > 0].dropna()

    out = pd.Series(pd.NaT, index=x.index, dtype="datetime64[ns]")

    if positive.empty:
        return out

    med = positive.median()
    valid = x_num > 0

    if 19000101 <= med <= 21001231:
        as_text = x_num.loc[valid].round().astype("Int64").astype(str).str.zfill(8)
        out.loc[valid] = pd.to_datetime(as_text, format="%Y%m%d", errors="coerce")
        return out

    if med > 1e17:
        unit = "ns"
    elif med > 1e14:
        unit = "us"
    elif med > 1e11:
        unit = "ms"
    else:
        unit = "s"

    dt = pd.to_datetime(x_num.loc[valid], unit=unit, errors="coerce", utc=True)
    out.loc[valid] = dt.dt.tz_convert(LOCAL_TIMEZONE).dt.tz_localize(None)

    return out


def parse_datetime_series(s: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(s):
        return parse_numeric_datetime_series(s)

    text = s.astype(str).str.strip()
    numeric = pd.to_numeric(text, errors="coerce")

    if numeric.notna().mean() >= 0.80:
        return parse_numeric_datetime_series(numeric)

    return pd.to_datetime(text, errors="coerce")


def normalize_kline_serial_to_daily(
    kline_df: pd.DataFrame,
    commodity: str,
    symbol: str,
    csv_path: Path,
) -> pd.DataFrame:
    """
    将 TqSdk 主力连续合约日线 K 线标准化为研究用 OHLCV。
    """
    if kline_df is None or len(kline_df) == 0:
        return pd.DataFrame()

    raw = kline_df.copy()
    raw.columns = [str(c).strip() for c in raw.columns]
    lower_map = {c.lower(): c for c in raw.columns}

    required = ["datetime", "open", "high", "low", "close", "volume"]
    missing = [c for c in required if c not in lower_map]
    if missing:
        raise ValueError(f"日线 K 线缺少字段：{missing}；当前字段：{list(raw.columns)}")

    df = pd.DataFrame()
    df["datetime"] = parse_datetime_series(raw[lower_map["datetime"]])
    df["date"] = df["datetime"].dt.normalize()

    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(raw[lower_map[c]], errors="coerce")

    df["commodity"] = commodity
    df["commodity_name"] = PRODUCTS[commodity].name
    df["main_symbol"] = symbol
    df["data_source"] = "TQ_MAIN_CONTINUOUS_KLINE_DAILY"
    df["csv_path"] = str(csv_path)

    start_ts = pd.Timestamp(max(PRODUCT_START_DATE.get(commodity, START_DATE), START_DATE))
    end_ts = pd.Timestamp(END_DATE)

    valid = (
        df["date"].notna()
        & (df["date"] >= start_ts)
        & (df["date"] <= end_ts)
        & (df[["open", "high", "low", "close"]] > 0).all(axis=1)
        & (df["volume"].fillna(-1) >= MIN_OHLCV_VOLUME)
        & (df["high"] >= df[["open", "close", "low"]].max(axis=1))
        & (df["low"] <= df[["open", "close", "high"]].min(axis=1))
    )

    df = df.loc[valid].copy()

    if df.empty:
        return pd.DataFrame()

    df = df.sort_values(["date", "datetime"]).drop_duplicates(["date"], keep="last")

    keep_cols = [
        "date", "datetime", "commodity", "commodity_name", "main_symbol", "data_source",
        "open", "high", "low", "close", "volume", "csv_path",
    ]

    return df[keep_cols].reset_index(drop=True)


def tq_wait_update_safe(api, deadline: int = 1) -> None:
    try:
        api.wait_update(deadline=deadline)
    except TypeError:
        api.wait_update()


def fetch_one_daily_kline_csv(
    api,
    commodity: str,
    symbol: str,
    csv_path: Path,
    overwrite: bool = False,
    data_length: int = DAILY_KLINE_DATA_LENGTH,
) -> Tuple[bool, str, int]:
    """
    直接抓取 TqSdk 主力连续合约日线 K 线并保存 CSV。
    """
    if csv_path.exists() and not overwrite:
        existing = pd.read_csv(csv_path)
        return True, "existing", int(len(existing))

    csv_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        klines = api.get_kline_serial(
            symbol,
            DAILY_KLINE_DUR_SEC,
            data_length=data_length,
        )

        if hasattr(api, "is_serial_ready"):
            wait_count = 0
            while not api.is_serial_ready(klines) and wait_count < 200:
                tq_wait_update_safe(api, deadline=1)
                wait_count += 1
        else:
            for _ in range(20):
                tq_wait_update_safe(api, deadline=1)

        daily_panel = normalize_kline_serial_to_daily(
            klines.copy(),
            commodity,
            symbol,
            csv_path,
        )

        if daily_panel.empty:
            return False, "日线 K 线为空或清洗后无有效 OHLCV", 0

        daily_panel.to_csv(csv_path, index=False)
        return True, "", int(len(daily_panel))

    except Exception as e:
        return False, str(e), 0


def download_main_continuous_for_products(
    products: Dict[str, ProductSpec] = PRODUCTS,
    overwrite: bool = OVERWRITE_RAW_CSV,
) -> pd.DataFrame:
    rows = []

    with closing(create_tq_api()) as api:
        for commodity, spec in products.items():
            csv_path = main_csv_path(commodity, spec.main_symbol)

            ok, err, rows_count = fetch_one_daily_kline_csv(
                api=api,
                commodity=commodity,
                symbol=spec.main_symbol,
                csv_path=csv_path,
                overwrite=overwrite,
            )

            rows.append({
                "commodity": commodity,
                "symbol": spec.main_symbol,
                "csv_path": str(csv_path),
                "ok": ok,
                "rows": rows_count,
                "error": err,
            })

            print(f"[主连日线] {commodity}: ok={ok}, rows={rows_count}, csv={csv_path}")

    report = pd.DataFrame(rows)
    report.to_csv(RESULT_DIR / "tq_main_continuous_daily_download_report.csv", index=False)
    return report


if RUN_TQ_DOWNLOAD:
    if has_tq_credentials():
        main_download_report = download_main_continuous_for_products()
        display(main_download_report)
    else:
        print(
            "RUN_TQ_DOWNLOAD=True，但尚未检测到天勤账号。"
            "请先在本地环境变量设置 TQ_USERNAME / TQ_PASSWORD，再重跑 Cell 04-05。"
        )
else:
    print(
        "RUN_TQ_DOWNLOAD=False。若本地 CSV 缺失且已设置天勤账号、已安装 tqsdk，"
        "后续 Cell 会自动抓取日线 K 线。"
    )


RUN_TQ_DOWNLOAD=False。若本地 CSV 缺失且已设置天勤账号、已安装 tqsdk，后续 Cell 会自动抓取日线 K 线。


In [11]:
def normalize_tq_daily_csv(csv_path: Path, commodity: str, symbol: str) -> pd.DataFrame:
    raw = pd.read_csv(csv_path)
    raw.columns = [str(c).strip() for c in raw.columns]
    lower_map = {c.lower(): c for c in raw.columns}

    dt_col = None
    for cand in ["datetime", "date", "time", "trading_day"]:
        if cand in lower_map:
            dt_col = lower_map[cand]
            break
    if dt_col is None:
        raise ValueError(f"{csv_path} 没有 datetime/date/time/trading_day 字段")

    col_map = {}
    for std in ["open", "high", "low", "close", "volume"]:
        if std in lower_map:
            col_map[lower_map[std]] = std

    missing = sorted(set(["open", "high", "low", "close", "volume"]) - set(col_map.values()))
    if missing:
        raise ValueError(f"{csv_path} 缺少 OHLCV 字段：{missing}")

    df = raw[[dt_col] + list(col_map.keys())].rename(columns={dt_col: "datetime", **col_map})
    df["datetime"] = parse_datetime_series(df["datetime"])
    df["date"] = df["datetime"].dt.normalize()

    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["commodity"] = commodity
    df["commodity_name"] = PRODUCTS[commodity].name
    df["main_symbol"] = symbol

    if "data_source" in lower_map:
        source_val = (
            str(raw[lower_map["data_source"]].dropna().iloc[0])
            if raw[lower_map["data_source"]].notna().any()
            else "TQ_MAIN_CONTINUOUS_KLINE_DAILY"
        )
    else:
        source_val = "TQ_MAIN_CONTINUOUS_KLINE_DAILY"

    df["data_source"] = source_val
    df["csv_path"] = str(csv_path)

    start_ts = pd.Timestamp(max(PRODUCT_START_DATE.get(commodity, START_DATE), START_DATE))
    end_ts = pd.Timestamp(END_DATE)

    valid = (
        df["date"].notna()
        & (df["date"] >= start_ts)
        & (df["date"] <= end_ts)
        & (df[["open", "high", "low", "close"]] > 0).all(axis=1)
        & (df["volume"].fillna(-1) >= MIN_OHLCV_VOLUME)
        & (df["high"] >= df[["open", "close", "low"]].max(axis=1))
        & (df["low"] <= df[["open", "close", "high"]].min(axis=1))
    )

    df = df.loc[valid].copy()

    keep_cols = [
        "date", "datetime", "commodity", "commodity_name", "main_symbol",
        "data_source", "open", "high", "low", "close", "volume", "csv_path",
    ]

    df = df[keep_cols]
    df = df.sort_values(["commodity", "date", "datetime"])
    df = df.drop_duplicates(["commodity", "date"], keep="last")

    return df.reset_index(drop=True)


def list_main_csvs(commodity: str) -> List[Path]:
    spec = PRODUCTS[commodity]
    expected = main_csv_path(commodity, spec.main_symbol)
    if expected.exists():
        return [expected]
    return sorted(
        RAW_MAIN_DIR.glob(f"{commodity}_*_daily.csv"),
        key=lambda p: (p.stat().st_mtime, p.name),
    )

In [12]:
def missing_required_commodities(df: pd.DataFrame) -> List[str]:
    present = set(df["commodity"].dropna().unique()) if "commodity" in df.columns and not df.empty else set()
    return [commodity for commodity in PRODUCTS if commodity not in present]


def load_main_continuous_panel() -> pd.DataFrame:
    frames = []
    errors = []

    for commodity, spec in PRODUCTS.items():
        paths = list_main_csvs(commodity)
        if not paths:
            print(f"[WARN] 未发现 {commodity} 的主力连续合约日线 CSV")
            continue
        path = paths[-1]
        try:
            df = normalize_tq_daily_csv(path, commodity, symbol=spec.main_symbol)
            if len(df) > 0:
                frames.append(df)
        except Exception as e:
            errors.append({"commodity": commodity, "csv_path": str(path), "error": str(e)})

    if errors:
        pd.DataFrame(errors).to_csv(RESULT_DIR / "load_main_continuous_errors.csv", index=False)
        print(f"[WARN] 主力连续合约日线 CSV 读取错误数量：{len(errors)}")

    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def load_research_panel() -> pd.DataFrame:
    main_panel = load_main_continuous_panel()
    missing = missing_required_commodities(main_panel)

    if missing and AUTO_DOWNLOAD_IF_MISSING:
        if has_tq_credentials():
            print(f"[AUTO] 主力连续合约日线 CSV 不完整，开始补齐：{missing}")
            report = download_main_continuous_for_products({c: PRODUCTS[c] for c in missing})
            display(report)
            main_panel = load_main_continuous_panel()
            missing = missing_required_commodities(main_panel)
        else:
            print(
                f"[AUTO] 主力连续合约日线 CSV 不完整，缺失 {missing}，且未检测到天勤账号环境变量。"
                "请在本地环境变量中设置 TQ_USERNAME / TQ_PASSWORD，并重新运行 Cell 04-07。"
            )

    if not missing:
        print(f"[OK] 已载入主力连续合约数据：{len(main_panel):,} 行，{main_panel['commodity'].nunique()} 个标的")
        return main_panel

    raise RuntimeError(
        f"未能载入完整 {len(PRODUCTS)} 个主力连续合约日线数据，缺失：{missing}。请处理以下任一项：\n"
        "1. 在本地环境变量中设置 TQ_USERNAME / TQ_PASSWORD，并重新运行 Cell 04-07；或\n"
        "2. 设置 RUN_TQ_DOWNLOAD=True，并从 Cell 03 开始重新运行；或\n"
        f"3. 将 {len(PRODUCTS)} 个日线 CSV 放入 {RAW_MAIN_DIR}/。"
    )


panel_raw = load_research_panel()
display(panel_raw.head())
display(panel_raw.groupby(["commodity", "main_symbol"]).agg(
    rows=("date", "size"),
    start=("date", "min"),
    end=("date", "max"),
))

[OK] 已载入主力连续合约数据：24,202 行，11 个标的


,date,datetime,commodity,commodity_name,main_symbol,data_source,open,high,low,close,volume,csv_path
0,2016-01-05,2016-01-05,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2327.0,2347.0,2325.0,2343.0,569527.0,main_continuous_daily_trend_momentum_reversal_...
1,2016-01-06,2016-01-06,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2345.0,2352.0,2327.0,2347.0,683608.0,main_continuous_daily_trend_momentum_reversal_...
2,2016-01-07,2016-01-07,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2341.0,2386.0,2339.0,2376.0,987863.0,main_continuous_daily_trend_momentum_reversal_...
3,2016-01-08,2016-01-08,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2375.0,2407.0,2363.0,2404.0,989187.0,main_continuous_daily_trend_momentum_reversal_...
4,2016-01-11,2016-01-11,M,豆粕,KQ.m@DCE.m,TQ_MAIN_CONTINUOUS_KLINE_DAILY,2407.0,2413.0,2377.0,2383.0,745035.0,main_continuous_daily_trend_momentum_reversal_...


,,rows,start,end
commodity,main_symbol,,,
AG,KQ.m@SHFE.ag,2545,2016-01-05,2026-06-30
CF,KQ.m@CZCE.CF,2545,2016-01-05,2026-06-30
EB,KQ.m@DCE.eb,1635,2019-09-26,2026-06-30
FG,KQ.m@CZCE.FG,2545,2016-01-05,2026-06-30
LH,KQ.m@DCE.lh,1324,2021-01-08,2026-06-30
M,KQ.m@DCE.m,2545,2016-01-05,2026-06-30
MA,KQ.m@CZCE.MA,2545,2016-01-05,2026-06-30
SA,KQ.m@CZCE.SA,1589,2019-12-06,2026-06-30
SP,KQ.m@SHFE.sp,1839,2018-11-27,2026-06-30


In [13]:
def clean_ohlcv_panel(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out = out.sort_values(["commodity", "date"]).reset_index(drop=True)

    for c in ["open", "high", "low", "close", "volume"]:
        out[c] = pd.to_numeric(out[c], errors="coerce")

    bad = (
        out[["open", "high", "low", "close"]].isna().any(axis=1)
        | (out[["open", "high", "low", "close"]] <= 0).any(axis=1)
        | (out["volume"].fillna(-1) < MIN_OHLCV_VOLUME)
        | (out["high"] < out[["open", "close", "low"]].max(axis=1))
        | (out["low"] > out[["open", "close", "high"]].min(axis=1))
    )

    out["ohlcv_bad"] = bad.astype(float)
    out = out.loc[~bad].copy()

    out["series_age"] = out.groupby("commodity").cumcount() + 1
    out["series_len"] = out.groupby("commodity")["date"].transform("size")
    out["series_remain"] = out["series_len"] - out["series_age"]
    out["series_start"] = out.groupby("commodity")["date"].transform("min")
    out["series_end"] = out.groupby("commodity")["date"].transform("max")
    out["data_reliability"] = "MAIN_CONTINUOUS_DIAGNOSTIC"

    return out.sort_values(["commodity", "date"]).reset_index(drop=True)


panel = clean_ohlcv_panel(panel_raw)
missing_after_cleaning = missing_required_commodities(panel)
if missing_after_cleaning:
    raise RuntimeError(f"OHLCV 清洗后缺失品种：{missing_after_cleaning}。请检查对应主力连续合约日线 CSV。")

display(panel.head())
display(panel.groupby(["commodity", "data_reliability"]).agg(
    rows=("date", "size"),
    start=("date", "min"),
    end=("date", "max"),
))

,date,datetime,commodity,commodity_name,main_symbol,data_source,open,high,low,close,volume,csv_path,ohlcv_bad,series_age,series_len,series_remain,series_start,series_end,data_reliability
0,2016-01-05,2016-01-05,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3306.0,3330.0,3294.0,3313.0,287111.0,main_continuous_daily_trend_momentum_reversal_...,0.0,1,2545,2544,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC
1,2016-01-06,2016-01-06,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3314.0,3321.0,3296.0,3314.0,204526.0,main_continuous_daily_trend_momentum_reversal_...,0.0,2,2545,2543,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC
2,2016-01-07,2016-01-07,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3322.0,3385.0,3308.0,3357.0,488186.0,main_continuous_daily_trend_momentum_reversal_...,0.0,3,2545,2542,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC
3,2016-01-08,2016-01-08,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3367.0,3390.0,3345.0,3356.0,379737.0,main_continuous_daily_trend_momentum_reversal_...,0.0,4,2545,2541,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC
4,2016-01-11,2016-01-11,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3345.0,3362.0,3322.0,3349.0,283036.0,main_continuous_daily_trend_momentum_reversal_...,0.0,5,2545,2540,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC


,,rows,start,end
commodity,data_reliability,,,
AG,MAIN_CONTINUOUS_DIAGNOSTIC,2545,2016-01-05,2026-06-30
CF,MAIN_CONTINUOUS_DIAGNOSTIC,2545,2016-01-05,2026-06-30
EB,MAIN_CONTINUOUS_DIAGNOSTIC,1635,2019-09-26,2026-06-30
FG,MAIN_CONTINUOUS_DIAGNOSTIC,2545,2016-01-05,2026-06-30
LH,MAIN_CONTINUOUS_DIAGNOSTIC,1324,2021-01-08,2026-06-30
M,MAIN_CONTINUOUS_DIAGNOSTIC,2545,2016-01-05,2026-06-30
MA,MAIN_CONTINUOUS_DIAGNOSTIC,2545,2016-01-05,2026-06-30
SA,MAIN_CONTINUOUS_DIAGNOSTIC,1589,2019-12-06,2026-06-30
SP,MAIN_CONTINUOUS_DIAGNOSTIC,1839,2018-11-27,2026-06-30


In [14]:
def nw_tstat(x: Sequence[float], lags: Optional[int] = None) -> Tuple[float, float]:
    arr = pd.Series(x).dropna().astype(float).to_numpy()
    n = len(arr)
    if n < 3:
        return np.nan, np.nan

    mean = float(np.mean(arr))
    demeaned = arr - mean

    if lags is None:
        lags = int(np.floor(4 * (n / 100) ** (2 / 9)))

    lags = max(0, min(int(lags), n - 1))

    gamma0 = float(np.dot(demeaned, demeaned) / n)
    lrv = gamma0

    for lag in range(1, lags + 1):
        cov = float(np.dot(demeaned[lag:], demeaned[:-lag]) / n)
        weight = 1.0 - lag / (lags + 1.0)
        lrv += 2.0 * weight * cov

    se = math.sqrt(max(lrv, 0) / n) if n > 0 else np.nan
    t = mean / se if se and se > 0 else np.nan

    return mean, t


def model_bootstrap_seed(
    label: str = "",
    k: Optional[int] = None,
    base_seed: int = RANDOM_SEED,
) -> int:
    """Return a deterministic model-specific seed for reproducible bootstrap draws."""
    key = f"{base_seed}|{label}|{k}".encode("utf-8")
    digest = hashlib.blake2b(key, digest_size=8).digest()
    return int.from_bytes(digest, byteorder="little", signed=False) % (2 ** 32)


def block_bootstrap_p_mean_le_zero(
    x: Sequence[float],
    block: int = BOOTSTRAP_BLOCK,
    n_boot: int = BOOTSTRAP_N,
    seed: int = RANDOM_SEED,
) -> float:
    """Block-bootstrap estimate of P(mean <= 0)."""
    arr = pd.Series(x).dropna().astype(float).to_numpy()
    n = len(arr)

    block = max(1, int(block))

    if n < max(10, block):
        return np.nan

    blocks = [arr[i:i + block] for i in range(0, n - block + 1)]
    if not blocks:
        return np.nan

    boot_rng = np.random.default_rng(seed)
    means = []
    need_blocks = int(math.ceil(n / block))

    for _ in range(n_boot):
        sample = np.concatenate([
            blocks[i]
            for i in boot_rng.integers(0, len(blocks), size=need_blocks)
        ])[:n]
        means.append(np.mean(sample))

    return float(np.mean(np.asarray(means) <= 0))


def summarize_directed_returns(
    directed_y: Sequence[float],
    raw_future_ret: Optional[Sequence[float]] = None,
    k: Optional[int] = None,
    label: str = "",
) -> Dict[str, float]:
    y_raw = pd.to_numeric(
        pd.Series(directed_y).replace([np.inf, -np.inf], np.nan),
        errors="coerce",
    )

    if raw_future_ret is not None:
        raw_raw = pd.to_numeric(
            pd.Series(raw_future_ret).replace([np.inf, -np.inf], np.nan),
            errors="coerce",
        )
        if len(y_raw) != len(raw_raw):
            raise ValueError(f"{label}: directed_y 与 raw_future_ret 长度不一致")
    else:
        raw_raw = None

    y = y_raw.dropna().astype(float)
    n = int(len(y))

    if n == 0:
        return {
            "label": label,
            "K": k,
            "n": 0,
            "mean_y": np.nan,
            "t_nw": np.nan,
            "nw_lags": np.nan,
            "hit_rate": np.nan,
            "boot_block": np.nan,
            "boot_p_mean_le_0": np.nan,
            "mean_raw": np.nan,
            "q05_y": np.nan,
            "q95_y": np.nan,
        }

    if raw_raw is not None:
        raw_aligned = raw_raw.loc[y.index]
        if raw_aligned.isna().any():
            raise ValueError(f"{label}: directed_y 有效样本对应的 raw_future_ret 存在缺失")
        raw = raw_aligned.astype(float)
    else:
        raw = pd.Series(dtype=float)

    k_eff = int(k) if k is not None and pd.notna(k) else 1

    auto_lags = int(np.floor(4 * (n / 100) ** (2 / 9)))
    overlap_lags = max(0, k_eff - 1)
    nw_lags = max(auto_lags, overlap_lags)
    nw_lags = max(0, min(int(nw_lags), n - 1))

    boot_block = max(BOOTSTRAP_BLOCK, k_eff)
    boot_seed = model_bootstrap_seed(label=label, k=k_eff, base_seed=RANDOM_SEED)

    _, t_nw = nw_tstat(y, lags=nw_lags)

    return {
        "label": label,
        "K": k,
        "n": n,
        "mean_y": float(y.mean()),
        "t_nw": float(t_nw) if pd.notna(t_nw) else np.nan,
        "nw_lags": int(nw_lags),
        "hit_rate": float((y > 0).mean()),
        "boot_block": int(boot_block),
        "boot_p_mean_le_0": block_bootstrap_p_mean_le_zero(y, block=boot_block, seed=boot_seed),
        "mean_raw": float(raw.mean()) if len(raw) else np.nan,
        "q05_y": float(y.quantile(0.05)),
        "q95_y": float(y.quantile(0.95)),
    }


def add_model_flags(scores: pd.DataFrame) -> pd.DataFrame:
    if scores.empty:
        return scores

    out = scores.copy()

    out["candidate_model"] = (
        (out["n"] >= MIN_MODEL_N)
        & (out["mean_y"] > 0)
        & (out["t_nw"] >= MIN_T_CAND)
        & (out["hit_rate"] > MIN_HIT_RATE)
    )

    out["confirmed_model"] = (
        out["candidate_model"]
        & (out["n"] >= MIN_CONFIRMED_N)
        & (out["t_nw"] >= MIN_T_CONF)
        & (out["hit_rate"] > MIN_HIT_RATE)
        & ((out["boot_p_mean_le_0"].isna()) | (out["boot_p_mean_le_0"] <= 0.10))
    )

    out["model_quality"] = np.select(
        [out["confirmed_model"], out["candidate_model"]],
        ["CONFIRMED", "CANDIDATE"],
        default="DIAGNOSTIC_ONLY",
    )

    return out


def zscore_by_commodity(df: pd.DataFrame, col: str) -> pd.Series:
    def _z(s):
        std = s.std(ddof=0)
        return (s - s.mean()) / std if pd.notna(std) and std > 0 else pd.Series(np.nan, index=s.index)

    return df.groupby("commodity")[col].transform(_z)


def pick_best_model(scores: pd.DataFrame) -> pd.DataFrame:
    if scores.empty:
        return pd.DataFrame()

    out = scores.copy()

    for c in ["confirmed_model", "candidate_model"]:
        if c not in out.columns:
            out[c] = False

    out = out.sort_values(
        ["commodity", "confirmed_model", "candidate_model", "t_nw", "mean_y", "hit_rate", "n"],
        ascending=[True, False, False, False, False, False, False],
    )

    return out.groupby("commodity", as_index=False).head(1).reset_index(drop=True)

In [16]:
def add_features_one_commodity(g: pd.DataFrame) -> pd.DataFrame:
    g = g.copy().sort_values("date").reset_index(drop=True)
    g["series_age"] = np.arange(1, len(g) + 1)
    g["series_len"] = len(g)
    g["series_remain"] = g["series_len"] - g["series_age"]

    g["ret_clean"] = np.log(g["close"] / g["close"].shift(1))
    g.loc[g["series_age"] == 1, "ret_clean"] = np.nan

    prev_close = g["close"].shift(1)
    tr1 = g["high"] - g["low"]
    tr2 = (g["high"] - prev_close).abs()
    tr3 = (g["low"] - prev_close).abs()
    g["tr"] = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    g.loc[g["series_age"] == 1, "tr"] = np.nan
    g["atr20"] = g["tr"].rolling(ATR_STOP_LOOKBACK, min_periods=max(5, ATR_STOP_LOOKBACK // 2)).mean()

    g["log_volume"] = np.log1p(g["volume"].clip(lower=0))

    g["sigma_roll20"] = g["ret_clean"].rolling(
        VOL_BASE_LOOKBACK,
        min_periods=max(5, VOL_BASE_LOOKBACK // 2),
    ).std(ddof=0)

    g["sigma_ewma"] = g["ret_clean"].ewm(
        alpha=1 - EWMA_LAMBDA,
        adjust=False,
        min_periods=5,
    ).std()

    g["sigma_daily_base"] = g["sigma_roll20"].combine_first(g["sigma_ewma"])
    g.loc[g["sigma_daily_base"] < MIN_SIGMA, "sigma_daily_base"] = np.nan

    g["pk_var_1"] = (np.log(g["high"] / g["low"]) ** 2) / (4 * np.log(2))
    g["pk_sigma_1"] = np.sqrt(g["pk_var_1"]).replace([np.inf, -np.inf], np.nan)

    g["std_ret_1"] = g["ret_clean"] / g["sigma_daily_base"]
    g["shock_flag"] = (g["std_ret_1"].abs() >= SHOCK_Z_THRESHOLD).astype(float)

    for h in sorted(set(H_RET_LIST + J_LIST)):
        g[f"R_{h}"] = g["ret_clean"].rolling(h, min_periods=h).sum()
        g[f"sigma_base_{h}"] = g["sigma_daily_base"] * math.sqrt(h)
        g[f"D_{h}"] = g[f"R_{h}"] / g[f"sigma_base_{h}"]

        volz_raw = (
            (g["log_volume"] - g["log_volume"].rolling(h, min_periods=h).mean())
            / g["log_volume"].rolling(h, min_periods=h).std(ddof=0)
        )
        g[f"VolZ_{h}"] = volz_raw

        past_std = g["std_ret_1"].shift(1)
        g[f"RAMOM_{h}"] = past_std.rolling(h, min_periods=h).sum()

    for k in K_LIST:
        g[f"fwd_ret_{k}"] = np.log(g["close"].shift(-k) / g["close"])
        g[f"valid_target_{k}"] = (g["series_remain"] >= k).astype(float)
        g.loc[g[f"valid_target_{k}"] != 1, f"fwd_ret_{k}"] = np.nan
        g[f"Y_{k}"] = g[f"fwd_ret_{k}"] / (g["sigma_daily_base"] * math.sqrt(k))

    return g.replace([np.inf, -np.inf], np.nan)


feature_panel = pd.concat(
    [add_features_one_commodity(g) for _, g in panel.groupby("commodity", sort=False)],
    ignore_index=True,
).reset_index(drop=True)

display(feature_panel.head())
display(feature_panel[["commodity", "date", "close", "ret_clean", "sigma_daily_base", "Y_5"]].tail())


,date,datetime,commodity,commodity_name,main_symbol,data_source,open,high,low,close,volume,csv_path,ohlcv_bad,series_age,series_len,series_remain,series_start,series_end,data_reliability,ret_clean,tr,atr20,log_volume,sigma_roll20,sigma_ewma,sigma_daily_base,pk_var_1,pk_sigma_1,std_ret_1,shock_flag,R_1,sigma_base_1,D_1,VolZ_1,RAMOM_1,R_2,sigma_base_2,D_2,VolZ_2,RAMOM_2,R_3,sigma_base_3,D_3,VolZ_3,RAMOM_3,R_5,sigma_base_5,D_5,VolZ_5,RAMOM_5,R_10,sigma_base_10,D_10,VolZ_10,RAMOM_10,R_20,sigma_base_20,D_20,VolZ_20,RAMOM_20,R_40,sigma_base_40,D_40,VolZ_40,RAMOM_40,R_60,sigma_base_60,D_60,VolZ_60,RAMOM_60,fwd_ret_1,valid_target_1,Y_1,fwd_ret_2,valid_target_2,Y_2,fwd_ret_3,valid_target_3,Y_3,fwd_ret_5,valid_target_5,Y_5,fwd_ret_10,valid_target_10,Y_10,fwd_ret_20,valid_target_20,Y_20
0,2016-01-05,2016-01-05,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3306.0,3330.0,3294.0,3313.0,287111.0,main_continuous_daily_trend_momentum_reversal_...,0.0,1,2545,2544,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC,NaN,NaN,NaN,12.567628,NaN,NaN,NaN,0.000043,0.006528,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000302,1.0,NaN,0.013194,1.0,NaN,0.012896,1.0,NaN,-0.007575,1.0,NaN,0.001207,1.0,NaN,0.007518,1.0,NaN
1,2016-01-06,2016-01-06,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3314.0,3321.0,3296.0,3314.0,204526.0,main_continuous_daily_trend_momentum_reversal_...,0.0,2,2545,2543,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC,0.000302,25.0,NaN,12.228455,NaN,NaN,NaN,0.000021,0.004538,NaN,0.0,0.000302,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.012892,1.0,NaN,0.012594,1.0,NaN,0.010506,1.0,NaN,-0.012143,1.0,NaN,0.000302,1.0,NaN,0.008114,1.0,NaN
2,2016-01-07,2016-01-07,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3322.0,3385.0,3308.0,3357.0,488186.0,main_continuous_daily_trend_momentum_reversal_...,0.0,3,2545,2542,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC,0.012892,77.0,NaN,13.098454,NaN,NaN,NaN,0.000191,0.013819,NaN,0.0,0.012892,NaN,NaN,NaN,NaN,0.013194,NaN,NaN,1.0,NaN,NaN,NaN,NaN,1.304173,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.000298,1.0,NaN,-0.002386,1.0,NaN,-0.020768,1.0,NaN,-0.013495,1.0,NaN,-0.007775,1.0,NaN,0.009192,1.0,NaN
3,2016-01-08,2016-01-08,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3367.0,3390.0,3345.0,3356.0,379737.0,main_continuous_daily_trend_momentum_reversal_...,0.0,4,2545,2541,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC,-0.000298,45.0,NaN,12.847237,NaN,NaN,NaN,0.000064,0.008025,NaN,0.0,-0.000298,NaN,NaN,NaN,NaN,0.012594,NaN,NaN,-1.0,NaN,0.012896,NaN,NaN,0.335135,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.002088,1.0,NaN,-0.020470,1.0,NaN,-0.024737,1.0,NaN,-0.018647,1.0,NaN,-0.009882,1.0,NaN,0.009785,1.0,NaN
4,2016-01-11,2016-01-11,AG,沪银,KQ.m@SHFE.ag,TQ_MAIN_CONTINUOUS_KLINE_DAILY,3345.0,3362.0,3322.0,3349.0,283036.0,main_continuous_daily_trend_momentum_reversal_...,0.0,5,2545,2540,2016-01-05,2026-06-30,MAIN_CONTINUOUS_DIAGNOSTIC,-0.002088,40.0,NaN,12.553333,NaN,NaN,NaN,0.000052,0.007188,NaN,0.0,-0.002088,NaN,NaN,NaN,NaN,-0.002386,NaN,NaN,-1.0,NaN,0.010506,NaN,NaN,-1.255431,NaN,NaN,NaN,NaN,-0.358963,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.018382,1.0,NaN,-0.022649,1.0,NaN,-0.011110,1.0,NaN,-0.014436,1.0,NaN,-0.007794,1.0,NaN,0.035493,1.0,NaN


,commodity,date,close,ret_clean,sigma_daily_base,Y_5
24197,V,2026-06-24,4490.0,0.010298,0.010578,NaN
24198,V,2026-06-25,4469.0,-0.004688,0.010426,NaN
24199,V,2026-06-26,4388.0,-0.018291,0.010006,NaN
24200,V,2026-06-29,4391.0,0.000683,0.010092,NaN
24201,V,2026-06-30,4354.0,-0.008462,0.009038,NaN


In [17]:
def feature_target_audit(df: pd.DataFrame) -> Dict[str, Dict]:
    """
    Feature / target leakage audit for the daily reversal pipeline.

    Main checks:
    1. Return-window features must not appear before enough historical returns exist.
    2. Deviation features D_h must follow the same timing rule as R_h.
    3. RAMOM_h uses shifted standardized returns, so it must not appear before h + 2 observations.
    4. Forward targets fwd_ret_k and Y_k must be empty when fewer than k observations remain.
    5. Cleaned OHLC rows must still satisfy positive-price and high/low consistency constraints.
    """
    report = {}

    ret_bad = {}
    deviation_bad = {}
    for h in H_RET_LIST:
        early_mask = df["series_age"] < h + 1
        ret_bad[h] = int(df.loc[early_mask, f"R_{h}"].notna().sum())
        deviation_bad[h] = int(df.loc[early_mask, f"D_{h}"].notna().sum())
    report["return_feature_non_nan_too_early"] = ret_bad
    report["deviation_feature_non_nan_too_early"] = deviation_bad

    reversal_aux_bad = {}
    for h in J_LIST:
        early_mask = df["series_age"] < h + 2
        reversal_aux_bad[h] = {
            "RAMOM": int(df.loc[early_mask, f"RAMOM_{h}"].notna().sum()),
        }
    report["reversal_aux_feature_non_nan_too_early"] = reversal_aux_bad

    target_bad = {}
    for k in K_LIST:
        end_mask = df["series_remain"] < k
        target_bad[k] = {
            "fwd_ret": int(df.loc[end_mask, f"fwd_ret_{k}"].notna().sum()),
            "Y": int(df.loc[end_mask, f"Y_{k}"].notna().sum()),
        }
    report["future_target_non_nan_beyond_series_end"] = target_bad

    bad_ohlc = int((
        (df["open"] <= 0)
        | (df["high"] <= 0)
        | (df["low"] <= 0)
        | (df["close"] <= 0)
        | (df["high"] < df[["open", "close", "low"]].max(axis=1))
        | (df["low"] > df[["open", "close", "high"]].min(axis=1))
    ).sum())
    report["bad_ohlc_after_cleaning"] = {"count": bad_ohlc}

    return report


audit_feature_target = feature_target_audit(feature_panel)
display(audit_feature_target)


def _audit_has_failure(x) -> bool:
    if isinstance(x, dict):
        return any(_audit_has_failure(v) for v in x.values())
    return x != 0


audit_failed = _audit_has_failure(audit_feature_target)

if audit_failed:
    raise RuntimeError(f"Feature / target audit failed: {audit_feature_target}")


{'return_feature_non_nan_too_early': {1: 0,
  2: 0,
  3: 0,
  5: 0,
  10: 0,
  20: 0,
  40: 0,
  60: 0},
 'deviation_feature_non_nan_too_early': {1: 0,
  2: 0,
  3: 0,
  5: 0,
  10: 0,
  20: 0,
  40: 0,
  60: 0},
 'reversal_aux_feature_non_nan_too_early': {2: {'RAMOM': 0},
  3: {'RAMOM': 0},
  5: {'RAMOM': 0},
  10: {'RAMOM': 0},
  20: {'RAMOM': 0},
  40: {'RAMOM': 0},
  60: {'RAMOM': 0}},
 'future_target_non_nan_beyond_series_end': {1: {'fwd_ret': 0, 'Y': 0},
  2: {'fwd_ret': 0, 'Y': 0},
  3: {'fwd_ret': 0, 'Y': 0},
  5: {'fwd_ret': 0, 'Y': 0},
  10: {'fwd_ret': 0, 'Y': 0},
  20: {'fwd_ret': 0, 'Y': 0}},
 'bad_ohlc_after_cleaning': {'count': 0}}

In [18]:
def reversal_score_table(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for commodity, g in df.groupby("commodity", sort=False):
        g = g.sort_values("date")

        for j in J_LIST:
            for k in K_LIST:
                if f"R_{j}" in g.columns and f"D_{j}" in g.columns:
                    side = -np.sign(g[f"R_{j}"])
                    mask = (
                        g[f"R_{j}"].notna()
                        & g[f"D_{j}"].notna()
                        & (g[f"R_{j}"].abs() > REV_SIGNAL_MIN_ABS)
                        & (g[f"D_{j}"].abs() >= TSREV_Z_THRESHOLD)
                        & (g[f"valid_target_{k}"] == 1)
                        & g[f"Y_{k}"].notna()
                        & side.notna()
                        & (side != 0)
                    )
                    if mask.sum() > 0:
                        stat = summarize_directed_returns(
                            side.loc[mask] * g.loc[mask, f"Y_{k}"],
                            raw_future_ret=side.loc[mask] * g.loc[mask, f"fwd_ret_{k}"],
                            k=k,
                            label=f"{commodity}_TSREV_J{j}_K{k}",
                        )
                        stat.update({"commodity": commodity, "J": j, "L": 0, "rev_type": "TSREV"})
                        rows.append(stat)

                if f"RAMOM_{j}" in g.columns:
                    side = -np.sign(g[f"RAMOM_{j}"])
                    mask = (
                        g[f"RAMOM_{j}"].notna()
                        & (g[f"RAMOM_{j}"].abs() >= RAREV_Z_THRESHOLD)
                        & (g[f"valid_target_{k}"] == 1)
                        & g[f"Y_{k}"].notna()
                        & side.notna()
                        & (side != 0)
                    )
                    if mask.sum() > 0:
                        stat = summarize_directed_returns(
                            side.loc[mask] * g.loc[mask, f"Y_{k}"],
                            raw_future_ret=side.loc[mask] * g.loc[mask, f"fwd_ret_{k}"],
                            k=k,
                            label=f"{commodity}_RAREV_J{j}_K{k}",
                        )
                        stat.update({"commodity": commodity, "J": j, "L": 0, "rev_type": "RAREV"})
                        rows.append(stat)

                if f"D_{j}" in g.columns:
                    side = -np.sign(g[f"D_{j}"])
                    mask = (
                        g[f"D_{j}"].notna()
                        & (g[f"D_{j}"].abs() >= FAST_REV_Z)
                        & (g[f"valid_target_{k}"] == 1)
                        & g[f"Y_{k}"].notna()
                        & side.notna()
                        & (side != 0)
                    )
                    if mask.sum() > 0:
                        stat = summarize_directed_returns(
                            side.loc[mask] * g.loc[mask, f"Y_{k}"],
                            raw_future_ret=side.loc[mask] * g.loc[mask, f"fwd_ret_{k}"],
                            k=k,
                            label=f"{commodity}_FastREV_J{j}_K{k}",
                        )
                        stat.update({"commodity": commodity, "J": j, "L": 0, "rev_type": "FastREV"})
                        rows.append(stat)

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out = add_model_flags(out)
    return out.sort_values(
        ["commodity", "confirmed_model", "candidate_model", "t_nw", "mean_y", "hit_rate", "n"],
        ascending=[True, False, False, False, False, False, False],
    ).reset_index(drop=True)


reversal_scores = reversal_score_table(feature_panel)
rev_best = pick_best_model(reversal_scores) if not reversal_scores.empty else pd.DataFrame()

display(reversal_scores.head(12))
display(rev_best)

reversal_scores.to_csv(RESULT_DIR / "reversal_model_scores.csv", index=False)


,label,K,n,mean_y,t_nw,nw_lags,hit_rate,boot_block,boot_p_mean_le_0,mean_raw,q05_y,q95_y,commodity,J,L,rev_type,candidate_model,confirmed_model,model_quality
0,AG_TSREV_J5_K5,5,828,0.071643,1.066113,6,0.543478,5,0.1425,0.002856,-2.062922,2.064997,AG,5,0,TSREV,True,False,CANDIDATE
1,AG_RAREV_J3_K5,5,1334,0.046735,1.015047,7,0.532984,5,0.1700,0.001909,-2.011763,1.914924,AG,3,0,RAREV,True,False,CANDIDATE
2,AG_FastREV_J10_K2,2,135,0.138023,0.962500,4,0.548148,5,0.1650,0.001390,-2.100474,2.124655,AG,10,0,FastREV,False,False,DIAGNOSTIC_ONLY
3,AG_FastREV_J10_K3,3,135,0.153647,0.956987,4,0.570370,5,0.2375,0.002925,-2.190228,2.207850,AG,10,0,FastREV,False,False,DIAGNOSTIC_ONLY
4,AG_FastREV_J10_K5,5,135,0.140383,0.854036,4,0.562963,5,0.2350,0.003102,-2.420371,2.269879,AG,10,0,FastREV,False,False,DIAGNOSTIC_ONLY
5,AG_FastREV_J5_K5,5,162,0.096779,0.698256,4,0.574074,5,0.2750,0.010092,-2.341618,2.241071,AG,5,0,FastREV,False,False,DIAGNOSTIC_ONLY
6,AG_RAREV_J3_K3,3,1336,0.029428,0.675252,7,0.540419,5,0.2500,0.000859,-2.028093,1.810887,AG,3,0,RAREV,False,False,DIAGNOSTIC_ONLY
7,AG_TSREV_J10_K2,2,840,0.038386,0.658272,6,0.545238,5,0.2950,0.000723,-2.043394,1.945027,AG,10,0,TSREV,False,False,DIAGNOSTIC_ONLY
8,AG_RAREV_J3_K10,10,1330,0.030935,0.602383,9,0.548872,10,0.2525,0.000699,-2.108414,1.851542,AG,3,0,RAREV,False,False,DIAGNOSTIC_ONLY
9,AG_TSREV_J10_K3,3,840,0.037252,0.546969,6,0.555952,5,0.2475,0.000678,-2.150970,1.921040,AG,10,0,TSREV,False,False,DIAGNOSTIC_ONLY


,label,K,n,mean_y,t_nw,nw_lags,hit_rate,boot_block,boot_p_mean_le_0,mean_raw,q05_y,q95_y,commodity,J,L,rev_type,candidate_model,confirmed_model,model_quality
0,AG_TSREV_J5_K5,5,828,0.071643,1.066113,6,0.543478,5,0.1425,0.002856,-2.062922,2.064997,AG,5,0,TSREV,True,False,CANDIDATE
1,CF_FastREV_J60_K20,20,109,0.579930,3.729932,19,0.678899,20,0.0000,0.011658,-1.342702,2.670312,CF,60,0,FastREV,True,False,CANDIDATE
2,EB_FastREV_J60_K20,20,57,0.606555,4.244212,19,0.789474,20,0.0000,0.049736,-1.133767,1.746607,EB,60,0,FastREV,False,False,DIAGNOSTIC_ONLY
3,FG_FastREV_J3_K10,10,125,0.217974,1.611736,9,0.608000,10,0.0750,0.011357,-1.724847,2.259377,FG,3,0,FastREV,True,False,CANDIDATE
4,LH_FastREV_J20_K20,20,103,0.219180,0.414436,19,0.533981,20,0.5475,0.016174,-2.855018,3.275473,LH,20,0,FastREV,False,False,DIAGNOSTIC_ONLY
5,M_TSREV_J60_K20,20,762,0.258449,2.112081,19,0.578740,20,0.0100,0.011440,-1.569398,2.157074,M,60,0,TSREV,True,True,CONFIRMED
6,MA_FastREV_J60_K10,10,96,0.797533,5.791973,9,0.781250,10,0.0000,0.037858,-0.657873,2.043660,MA,60,0,FastREV,True,False,CANDIDATE
7,SA_RAREV_J60_K20,20,1410,0.194813,1.671794,19,0.556738,20,0.0550,0.016259,-2.024724,2.583982,SA,60,0,RAREV,True,False,CANDIDATE
8,SP_FastREV_J3_K10,10,106,0.190347,1.049823,9,0.537736,10,0.1025,0.004858,-1.480153,2.438886,SP,3,0,FastREV,True,False,CANDIDATE
9,SR_TSREV_J2_K10,10,769,0.108983,2.400222,9,0.530559,10,0.0050,0.002553,-1.630135,1.958589,SR,2,0,TSREV,True,True,CONFIRMED


In [20]:
REV_ESTABLISH_CONFIRM_DAYS = max(1, int(globals().get("REV_ESTABLISH_CONFIRM_DAYS", 2)))


def consecutive_age(active: pd.Series) -> pd.Series:
    active = active.fillna(0).astype(bool)
    group = (~active).cumsum()
    return active.astype(int).groupby(group).cumsum()


def is_usable_reversal_model(row) -> bool:
    def _as_bool(x) -> bool:
        return bool(x) if pd.notna(x) else False

    quality = str(row.get("model_quality", "DIAGNOSTIC_ONLY")).upper()
    return (
        _as_bool(row.get("confirmed_model", False))
        or _as_bool(row.get("candidate_model", False))
        or quality in {"CONFIRMED", "CANDIDATE"}
    )


def add_reversal_state_columns(df: pd.DataFrame, best: pd.DataFrame) -> pd.DataFrame:
    out = df.copy().sort_values(["commodity", "date"]).reset_index(drop=True)

    for col in [
        "rev_best_type", "rev_confirmed_model", "rev_candidate_model",
        "rev_model_quality", "rev_model_scope",
        "REVModelUsable", "J_rev_star", "L_rev_star", "K_rev_eval", "REVSource",
        "REVSide", "REVSignal", "REVValidatedSignal", "REVVolumeDivergence",
        "REVValidatedAge", "ReversalEnd_signal", "ReversalEnd_flip",
        "ReversalEnd_fast_absorbed", "ReversalEnd",
    ]:
        out[col] = np.nan

    out["rev_best_type"] = pd.Series(pd.NA, index=out.index, dtype="object")
    out["rev_model_quality"] = pd.Series("NO_MODEL", index=out.index, dtype="object")
    out["rev_model_scope"] = pd.Series("NO_MODEL", index=out.index, dtype="object")  # backward-compatible alias
    out["REVSource"] = pd.Series("NONE", index=out.index, dtype="object")

    out["rev_confirmed_model"] = False
    out["rev_candidate_model"] = False
    out["REVModelUsable"] = False

    out["REVSide"] = np.nan
    out["REVSignal"] = 0.0
    out["REVValidatedSignal"] = 0.0

    if best is not None and not best.empty:
        for _, row in best.iterrows():
            commodity = row["commodity"]
            j = int(row["J"])
            L = int(row.get("L", 0))
            k = int(row["K"])
            rtype = row["rev_type"]
            model_quality = row.get("model_quality", "DIAGNOSTIC_ONLY")

            m = out["commodity"] == commodity

            out.loc[m, "rev_best_type"] = rtype
            out.loc[m, "J_rev_star"] = j
            out.loc[m, "L_rev_star"] = L
            out.loc[m, "K_rev_eval"] = k

            out.loc[m, "rev_confirmed_model"] = bool(row.get("confirmed_model", False))
            out.loc[m, "rev_candidate_model"] = bool(row.get("candidate_model", False))
            out.loc[m, "rev_model_quality"] = model_quality
            out.loc[m, "rev_model_scope"] = model_quality  # compatibility with older downstream cells
            out.loc[m, "REVModelUsable"] = is_usable_reversal_model(row)

            if rtype == "TSREV" and f"R_{j}" in out.columns and f"D_{j}" in out.columns:
                sig = (
                    m
                    & out[f"R_{j}"].notna()
                    & out[f"D_{j}"].notna()
                    & (out[f"R_{j}"].abs() > REV_SIGNAL_MIN_ABS)
                    & (out[f"D_{j}"].abs() >= TSREV_Z_THRESHOLD)
                )
                out.loc[sig, "REVSide"] = -np.sign(out.loc[sig, f"R_{j}"])
                out.loc[sig, "REVSignal"] = 1.0
                out.loc[sig, "REVSource"] = "TSREV"

            elif rtype == "RAREV" and f"RAMOM_{j}" in out.columns:
                sig = (
                    m
                    & out[f"RAMOM_{j}"].notna()
                    & (out[f"RAMOM_{j}"].abs() >= RAREV_Z_THRESHOLD)
                )
                out.loc[sig, "REVSide"] = -np.sign(out.loc[sig, f"RAMOM_{j}"])
                out.loc[sig, "REVSignal"] = 1.0
                out.loc[sig, "REVSource"] = "RAREV"

            elif rtype == "FastREV" and f"D_{j}" in out.columns:
                sig = (
                    m
                    & out[f"D_{j}"].notna()
                    & (out[f"D_{j}"].abs() >= FAST_REV_Z)
                )
                out.loc[sig, "REVSide"] = -np.sign(out.loc[sig, f"D_{j}"])
                out.loc[sig, "REVSignal"] = 1.0
                out.loc[sig, "REVSource"] = "FastREV"

    out["REVVolumeDivergence"] = 0.0
    for j in J_LIST:
        if f"R_{j}" in out.columns and f"VolZ_{j}" in out.columns:
            out[f"VolumeDivergence_{j}"] = (
                np.sign(out[f"R_{j}"]) * np.sign(out[f"VolZ_{j}"]) < 0
            ).astype(float)

            m = out["J_rev_star"] == j
            out.loc[m, "REVVolumeDivergence"] = out.loc[m, f"VolumeDivergence_{j}"]

    out.loc[(out["REVSignal"] == 1) & (~out["REVSide"].isin([-1, 1])), "REVSignal"] = 0.0
    out.loc[out["REVSignal"] != 1, "REVSide"] = np.nan

    raw_valid = (
        (out["REVSignal"] == 1)
        & (out["REVModelUsable"] == True)
        & out["REVSide"].isin([-1, 1])
    )
    out["_REVValidRaw"] = raw_valid.astype(float)

    frames = []
    for _, g in out.groupby("commodity", sort=False):
        g = g.copy().sort_values("date").reset_index(drop=True)

        raw_active = g["_REVValidRaw"] == 1
        side = g["REVSide"]

        block = ((~raw_active) | (side != side.shift(1))).cumsum()
        run_len = raw_active.astype(int).groupby(block).cumsum()
        active = raw_active & (run_len >= REV_ESTABLISH_CONFIRM_DAYS)

        g["REVValidatedSignal"] = active.astype(float)
        g["REVValidatedAge"] = consecutive_age(active).where(active, 0)

        prev_active = active.shift(1).fillna(False)
        prev_side = g["REVSide"].shift(1)
        prev_source = g["REVSource"].shift(1)

        # With REV_ESTABLISH_CONFIRM_DAYS >= 2, a same-day raw side flip is not yet
        # validated as a new episode. It should still close the previous validated
        # episode as FLIP rather than being misclassified as SIGNAL_LOSS.
        raw_side_changed = (
            prev_active
            & raw_active
            & prev_side.notna()
            & side.notna()
            & (np.sign(side) != np.sign(prev_side))
        )
        raw_signal_lost = prev_active & (~raw_active)

        fast_absorbed = pd.Series(False, index=g.index)
        j_star = pd.to_numeric(g["J_rev_star"], errors="coerce")

        for j in J_LIST:
            if f"D_{j}" in g.columns:
                use_j = j_star.eq(j)
                fast_absorbed = fast_absorbed | (
                    prev_active
                    & use_j
                    & (prev_source == "FastREV")
                    & (g[f"D_{j}"].abs() < 1)
                )

        # Make end reasons mutually exclusive for cleaner episode interpretation.
        # Priority: FLIP > FAST_ABSORBED > SIGNAL_LOSS.
        signal_lost_clean = raw_signal_lost & (~raw_side_changed) & (~fast_absorbed)

        g["ReversalEnd_flip"] = raw_side_changed.astype(float)
        g["ReversalEnd_fast_absorbed"] = fast_absorbed.astype(float)
        g["ReversalEnd_signal"] = signal_lost_clean.astype(float)

        g["ReversalEnd"] = (
            (g["ReversalEnd_signal"] == 1)
            | (g["ReversalEnd_flip"] == 1)
            | (g["ReversalEnd_fast_absorbed"] == 1)
        ).astype(float)

        frames.append(g)

    return (
        pd.concat(frames, ignore_index=True)
        .drop(columns=["_REVValidRaw"], errors="ignore")
        .replace([np.inf, -np.inf], np.nan)
    )


panel_state = add_reversal_state_columns(feature_panel, rev_best)

display(panel_state[[
    "date", "commodity", "rev_best_type", "rev_model_quality", "REVModelUsable",
    "J_rev_star", "K_rev_eval", "REVSource", "REVSide", "REVSignal",
    "REVValidatedSignal", "REVVolumeDivergence", "REVValidatedAge", "ReversalEnd",
]].tail(32))

,date,commodity,rev_best_type,rev_model_quality,REVModelUsable,J_rev_star,K_rev_eval,REVSource,REVSide,REVSignal,REVValidatedSignal,REVVolumeDivergence,REVValidatedAge,ReversalEnd
24170,2026-05-15,V,TSREV,CANDIDATE,True,60.0,20.0,NONE,NaN,0.0,0.0,1.0,0,0.0
24171,2026-05-18,V,TSREV,CANDIDATE,True,60.0,20.0,NONE,NaN,0.0,0.0,1.0,0,0.0
24172,2026-05-19,V,TSREV,CANDIDATE,True,60.0,20.0,NONE,NaN,0.0,0.0,1.0,0,0.0
24173,2026-05-20,V,TSREV,CANDIDATE,True,60.0,20.0,NONE,NaN,0.0,0.0,0.0,0,0.0
24174,2026-05-21,V,TSREV,CANDIDATE,True,60.0,20.0,NONE,NaN,0.0,0.0,0.0,0,0.0
24175,2026-05-22,V,TSREV,CANDIDATE,True,60.0,20.0,NONE,NaN,0.0,0.0,0.0,0,0.0
24176,2026-05-25,V,TSREV,CANDIDATE,True,60.0,20.0,NONE,NaN,0.0,0.0,0.0,0,0.0
24177,2026-05-26,V,TSREV,CANDIDATE,True,60.0,20.0,NONE,NaN,0.0,0.0,0.0,0,0.0
24178,2026-05-27,V,TSREV,CANDIDATE,True,60.0,20.0,NONE,NaN,0.0,0.0,1.0,0,0.0
24179,2026-05-28,V,TSREV,CANDIDATE,True,60.0,20.0,NONE,NaN,0.0,0.0,1.0,0,0.0


In [21]:
REV_SURVIVAL_HORIZONS = sorted(set(globals().get("REV_SURVIVAL_HORIZONS", K_LIST)))
REV_DURATION_MIN_COMPLETED = globals().get("REV_DURATION_MIN_COMPLETED", 5)

# Duration metrics are measured in validated-active observations within reversal episodes.
# They are not calendar days and should not be read as guaranteed trading holding periods.
REV_DURATION_UNIT = "validated_active_observations"
REV_DURATION_DEFINITION = (
    "Duration / remaining-life metrics count validated-active observations within completed "
    "reversal episodes, not calendar days or guaranteed holding periods."
)


def _rev_side_label(side) -> str:
    if pd.isna(side) or side == 0:
        return "NONE"
    return "UP" if side > 0 else "DOWN"


def _rev_end_reason(row: pd.Series) -> str:
    if bool(row.get("ReversalEnd_flip", False)):
        return "FLIP"
    if bool(row.get("ReversalEnd_fast_absorbed", False)):
        return "FAST_ABSORBED"
    if bool(row.get("ReversalEnd_signal", False)):
        return "SIGNAL_LOSS"
    return "UNFLAGGED"


def build_reversal_episodes(state: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    state = state.copy()

    # Keep Cell 14 compatible with both the newer accurate name and the older alias.
    if "rev_model_quality" not in state.columns and "rev_model_scope" in state.columns:
        state["rev_model_quality"] = state["rev_model_scope"]
    if "rev_model_scope" not in state.columns and "rev_model_quality" in state.columns:
        state["rev_model_scope"] = state["rev_model_quality"]

    required = [
        "date", "commodity", "series_remain", "REVSource", "REVSide", "REVSignal",
        "REVValidatedSignal", "ReversalEnd", "rev_best_type", "REVModelUsable",
        "J_rev_star", "L_rev_star", "K_rev_eval", "rev_model_quality", "rev_model_scope",
    ]
    missing = [c for c in required if c not in state.columns]
    if missing:
        raise RuntimeError(f"Reversal state is missing columns: {missing}")

    out = state.copy().sort_values(["commodity", "date"]).reset_index(drop=True)

    for col in [
        "REVEpisodeID", "REVEpisodeSide", "REVEpisodeSideLabel", "REVEpisodeSource",
        "REVValidatedAge", "REVEpisodeStartDate", "REVEpisodeEndDate", "REVEpisodeEndReason",
        "REVEpisodeCensored", "REVObsToEnd",
    ]:
        out[col] = pd.NA

    episodes = []

    def _source_abs_signal(seg: pd.DataFrame) -> pd.Series:
        vals = pd.Series(np.nan, index=seg.index, dtype=float)

        for _, row in seg.iterrows():
            j = row.get("J_rev_star", np.nan)
            rtype = row.get("REVSource", "NONE")

            if pd.isna(j):
                continue

            j = int(j)

            if rtype == "TSREV" and f"R_{j}" in seg.columns:
                vals.loc[row.name] = abs(row[f"R_{j}"])
            elif rtype == "RAREV" and f"RAMOM_{j}" in seg.columns:
                vals.loc[row.name] = abs(row[f"RAMOM_{j}"])
            elif rtype == "FastREV" and f"D_{j}" in seg.columns:
                vals.loc[row.name] = abs(row[f"D_{j}"])

        return vals

    def close_episode(g, meta, end_row, end_reason, censored):
        idxs = meta["idxs"]
        if not idxs:
            return

        n = len(idxs)
        end_date = pd.NaT if censored else end_row["date"]

        for age, local_idx in enumerate(idxs, start=1):
            global_idx = g.loc[local_idx, "_global_idx"]

            out.loc[global_idx, "REVEpisodeID"] = meta["episode_id"]
            out.loc[global_idx, "REVEpisodeSide"] = meta["side"]
            out.loc[global_idx, "REVEpisodeSideLabel"] = _rev_side_label(meta["side"])
            out.loc[global_idx, "REVEpisodeSource"] = meta["source"]
            out.loc[global_idx, "REVValidatedAge"] = age
            out.loc[global_idx, "REVEpisodeStartDate"] = meta["start_date"]
            out.loc[global_idx, "REVEpisodeEndDate"] = end_date
            out.loc[global_idx, "REVEpisodeEndReason"] = end_reason
            out.loc[global_idx, "REVEpisodeCensored"] = bool(censored)

            if not censored:
                # REVObsToEnd measures remaining validated-active observations in the episode.
                out.loc[global_idx, "REVObsToEnd"] = n - age + 1

        seg = g.loc[idxs]

        episodes.append({
            "commodity": meta["commodity"],
            "rev_episode_id": meta["episode_id"],
            "rev_side": meta["side"],
            "rev_side_label": _rev_side_label(meta["side"]),
            "rev_source": meta["source"],
            "established_date": meta["start_date"],
            "last_active_date": seg["date"].max(),
            "end_signal_date": end_date,
            "end_reason": end_reason,
            "active_duration_obs": n,
            "duration_to_end_signal_obs": n if not censored else np.nan,
            "is_censored": bool(censored),
            "ever_volume_divergence": (
                bool((seg["REVVolumeDivergence"] == 1).any())
                if "REVVolumeDivergence" in seg
                else False
            ),
            "max_abs_source_signal": float(_source_abs_signal(seg).max()),
            "mean_abs_source_signal": float(_source_abs_signal(seg).mean()),
            "J_rev_star": seg["J_rev_star"].dropna().iloc[-1] if seg["J_rev_star"].notna().any() else np.nan,
            "L_rev_star": seg["L_rev_star"].dropna().iloc[-1] if seg["L_rev_star"].notna().any() else np.nan,
            "K_rev_eval": seg["K_rev_eval"].dropna().iloc[-1] if seg["K_rev_eval"].notna().any() else np.nan,
            "rev_best_type": seg["rev_best_type"].dropna().iloc[-1] if seg["rev_best_type"].notna().any() else "NO_MODEL",
            "rev_model_quality": (
                seg["rev_model_quality"].dropna().iloc[-1]
                if seg["rev_model_quality"].notna().any()
                else "NO_MODEL"
            ),
            "rev_model_scope": (
                seg["rev_model_scope"].dropna().iloc[-1]
                if seg["rev_model_scope"].notna().any()
                else "NO_MODEL"
            ),
            "duration_unit": REV_DURATION_UNIT,
            "duration_definition": REV_DURATION_DEFINITION,
        })

    for commodity, g0 in out.groupby("commodity", sort=False):
        g = g0.copy().reset_index().rename(columns={"index": "_global_idx"})
        open_ep = None
        ep_no = 0

        for i, row in g.iterrows():
            active = bool(row.get("REVValidatedSignal", 0) == 1)
            side = row.get("REVSide", np.nan)
            valid_side = pd.notna(side) and side != 0

            if open_ep is not None:
                should_close = (
                    bool(row.get("ReversalEnd", 0) == 1)
                    or (not active)
                    or (not valid_side)
                    or (np.sign(side) != np.sign(open_ep["side"]))
                )

                if should_close:
                    close_episode(g, open_ep, row, _rev_end_reason(row), censored=False)
                    open_ep = None

            if open_ep is None and active and valid_side:
                ep_no += 1
                open_ep = {
                    "commodity": commodity,
                    "episode_id": f"{commodity}_REV_{ep_no:04d}",
                    "side": float(np.sign(side)),
                    "source": row.get("REVSource", "NONE"),
                    "start_date": row["date"],
                    "idxs": [],
                }

            if open_ep is not None and active and valid_side and (np.sign(side) == np.sign(open_ep["side"])):
                open_ep["idxs"].append(i)

        if open_ep is not None:
            close_episode(g, open_ep, g.iloc[-1], "END_OF_SAMPLE", censored=True)

    return (
        out.replace([np.inf, -np.inf], np.nan),
        pd.DataFrame(episodes).replace([np.inf, -np.inf], np.nan),
    )


def summarize_reversal_durations(episodes: pd.DataFrame) -> pd.DataFrame:
    if episodes.empty:
        return pd.DataFrame()

    rows = []

    for keys, g in episodes.groupby(["commodity", "rev_side_label", "rev_source"], sort=False):
        commodity, side_label, source = keys
        ended = g.loc[~g["is_censored"].astype(bool)].copy()

        rows.append({
            "commodity": commodity,
            "rev_side_label": side_label,
            "rev_source": source,
            "n_episodes": int(len(g)),
            "n_ended": int(len(ended)),
            "n_censored": int(g["is_censored"].sum()),
            "volume_divergence_rate": float(g["ever_volume_divergence"].mean()) if len(g) else np.nan,
            "mean_active_duration_obs": float(ended["active_duration_obs"].mean()) if len(ended) else np.nan,
            "median_active_duration_obs": float(ended["active_duration_obs"].median()) if len(ended) else np.nan,
            "q25_active_duration_obs": float(ended["active_duration_obs"].quantile(0.25)) if len(ended) else np.nan,
            "q75_active_duration_obs": float(ended["active_duration_obs"].quantile(0.75)) if len(ended) else np.nan,
            "mean_duration_to_end_signal_obs": (
                float(ended["duration_to_end_signal_obs"].mean()) if len(ended) else np.nan
            ),
            "median_duration_to_end_signal_obs": (
                float(ended["duration_to_end_signal_obs"].median()) if len(ended) else np.nan
            ),
            "duration_unit": REV_DURATION_UNIT,
            "duration_definition": REV_DURATION_DEFINITION,
        })

    return pd.DataFrame(rows)


def reversal_survival_by_horizon(panel_duration: pd.DataFrame) -> pd.DataFrame:
    active = panel_duration.loc[panel_duration["REVEpisodeID"].notna()].copy()

    if active.empty:
        return pd.DataFrame()

    rows = []

    for keys, g in active.groupby(["commodity", "REVEpisodeSideLabel", "REVEpisodeSource"], sort=False):
        commodity, side_label, source = keys

        for h in REV_SURVIVAL_HORIZONS:
            known = g["REVObsToEnd"].notna() | (g["series_remain"] >= h)
            gg = g.loc[known]

            if gg.empty:
                continue

            obs_to_end = pd.to_numeric(gg["REVObsToEnd"], errors="coerce")
            ended = obs_to_end.notna() & (obs_to_end <= h)

            rows.append({
                "commodity": commodity,
                "rev_side_label": side_label,
                "rev_source": source,
                "horizon_obs": h,
                "n_state_obs": int(len(gg)),
                "n_end_within_horizon_obs": int(ended.sum()),
                "p_end_within_horizon_obs": float(ended.mean()),
                "p_survive_beyond_horizon_obs": float(1 - ended.mean()),
                "duration_unit": REV_DURATION_UNIT,
                "duration_definition": REV_DURATION_DEFINITION,
            })

    return pd.DataFrame(rows)


def current_reversal_life_report(panel_duration: pd.DataFrame, episodes: pd.DataFrame) -> pd.DataFrame:
    latest = panel_duration.sort_values(["commodity", "date"]).groupby("commodity", sort=False).tail(1)

    completed = (
        episodes.loc[~episodes.get("is_censored", pd.Series(dtype=bool)).astype(bool)].copy()
        if not episodes.empty
        else pd.DataFrame()
    )

    rows = []

    for _, row in latest.iterrows():
        commodity = row["commodity"]
        active = bool(row.get("REVValidatedSignal", 0) == 1)
        side_label = row.get("REVEpisodeSideLabel", "NONE") if active else "NONE"
        source = row.get("REVEpisodeSource", "NONE") if active else "NONE"
        age = row.get("REVValidatedAge", np.nan)

        out = {
            "commodity": commodity,
            "date": row["date"],
            "reversal_active_now": active,
            "rev_side_label": side_label,
            "rev_source": source,
            "rev_validated_age_obs": age,
            "reversal_end_confirmed": bool(row.get("ReversalEnd", 0) == 1),
            "n_comparable_completed": 0,
            "duration_reliability": "NO_ACTIVE_REVERSAL" if not active else "INSUFFICIENT",
            "duration_unit": REV_DURATION_UNIT,
            "duration_definition": REV_DURATION_DEFINITION,
            "mean_remaining_obs": np.nan,
            "median_remaining_obs": np.nan,
        }

        for h in REV_SURVIVAL_HORIZONS:
            out[f"p_end_within_{h}obs"] = np.nan

        if active and pd.notna(age) and not completed.empty:
            comp = completed.loc[
                (completed["commodity"] == commodity)
                & (completed["rev_side_label"] == side_label)
                & (completed["rev_source"] == source)
                & (completed["active_duration_obs"] >= age)
            ].copy()

            remaining = comp["duration_to_end_signal_obs"] - (float(age) - 1)
            remaining = remaining.loc[remaining >= 1].astype(float)

            out["n_comparable_completed"] = int(len(remaining))

            if len(remaining) >= REV_DURATION_MIN_COMPLETED:
                out["duration_reliability"] = "OK"
                out["mean_remaining_obs"] = float(remaining.mean())
                out["median_remaining_obs"] = float(remaining.median())

                for h in REV_SURVIVAL_HORIZONS:
                    out[f"p_end_within_{h}obs"] = float((remaining <= h).mean())

        rows.append(out)

    return pd.DataFrame(rows)


panel_state_duration, reversal_episode_table = build_reversal_episodes(panel_state)
reversal_duration_summary = summarize_reversal_durations(reversal_episode_table)
reversal_survival_table = reversal_survival_by_horizon(panel_state_duration)
current_reversal_life_report_df = current_reversal_life_report(panel_state_duration, reversal_episode_table)

display(reversal_episode_table.tail(12))
display(reversal_duration_summary)
display(reversal_survival_table.head(20))
display(current_reversal_life_report_df)

panel_state_duration.to_csv(RESULT_DIR / "reversal_panel_state_duration.csv", index=False)
reversal_episode_table.to_csv(RESULT_DIR / "reversal_episode_table.csv", index=False)
reversal_duration_summary.to_csv(RESULT_DIR / "reversal_duration_summary.csv", index=False)
reversal_survival_table.to_csv(RESULT_DIR / "reversal_survival_table.csv", index=False)
current_reversal_life_report_df.to_csv(RESULT_DIR / "reversal_current_life_report.csv", index=False)

,commodity,rev_episode_id,rev_side,rev_side_label,rev_source,established_date,last_active_date,end_signal_date,end_reason,active_duration_obs,duration_to_end_signal_obs,is_censored,ever_volume_divergence,max_abs_source_signal,mean_abs_source_signal,J_rev_star,L_rev_star,K_rev_eval,rev_best_type,rev_model_quality,rev_model_scope,duration_unit,duration_definition
675,V,V_REV_0054,-1.0,DOWN,TSREV,2024-05-21,2024-05-31,2024-06-03,SIGNAL_LOSS,9,9.0,False,False,0.101600,0.091910,60.0,0.0,20.0,TSREV,CANDIDATE,CANDIDATE,validated_active_observations,Duration / remaining-life metrics count valida...
676,V,V_REV_0055,-1.0,DOWN,TSREV,2024-07-03,2024-07-03,2024-07-04,SIGNAL_LOSS,1,1.0,False,True,0.055728,0.055728,60.0,0.0,20.0,TSREV,CANDIDATE,CANDIDATE,validated_active_observations,Duration / remaining-life metrics count valida...
677,V,V_REV_0056,1.0,UP,TSREV,2024-08-07,2024-09-26,2024-09-27,SIGNAL_LOSS,35,35.0,False,True,0.181104,0.137059,60.0,0.0,20.0,TSREV,CANDIDATE,CANDIDATE,validated_active_observations,Duration / remaining-life metrics count valida...
678,V,V_REV_0057,1.0,UP,TSREV,2024-11-28,2024-12-11,2024-12-12,SIGNAL_LOSS,10,10.0,False,True,0.091219,0.076991,60.0,0.0,20.0,TSREV,CANDIDATE,CANDIDATE,validated_active_observations,Duration / remaining-life metrics count valida...
679,V,V_REV_0058,1.0,UP,TSREV,2024-12-16,2024-12-16,2024-12-17,SIGNAL_LOSS,1,1.0,False,False,0.066455,0.066455,60.0,0.0,20.0,TSREV,CANDIDATE,CANDIDATE,validated_active_observations,Duration / remaining-life metrics count valida...
680,V,V_REV_0059,1.0,UP,TSREV,2025-05-29,2025-06-03,2025-06-04,SIGNAL_LOSS,3,3.0,False,True,0.092941,0.089848,60.0,0.0,20.0,TSREV,CANDIDATE,CANDIDATE,validated_active_observations,Duration / remaining-life metrics count valida...
681,V,V_REV_0060,1.0,UP,TSREV,2025-10-15,2025-10-15,2025-10-16,SIGNAL_LOSS,1,1.0,False,False,0.061768,0.061768,60.0,0.0,20.0,TSREV,CANDIDATE,CANDIDATE,validated_active_observations,Duration / remaining-life metrics count valida...
682,V,V_REV_0061,1.0,UP,TSREV,2025-10-22,2025-12-16,2025-12-17,SIGNAL_LOSS,40,40.0,False,True,0.146948,0.096008,60.0,0.0,20.0,TSREV,CANDIDATE,CANDIDATE,validated_active_observations,Duration / remaining-life metrics count valida...
683,V,V_REV_0062,-1.0,DOWN,TSREV,2026-02-25,2026-02-25,2026-02-26,SIGNAL_LOSS,1,1.0,False,False,0.107759,0.107759,60.0,0.0,20.0,TSREV,CANDIDATE,CANDIDATE,validated_active_observations,Duration / remaining-life metrics count valida...
684,V,V_REV_0063,-1.0,DOWN,TSREV,2026-03-09,2026-03-09,2026-03-10,SIGNAL_LOSS,1,1.0,False,False,0.185400,0.185400,60.0,0.0,20.0,TSREV,CANDIDATE,CANDIDATE,validated_active_observations,Duration / remaining-life metrics count valida...


,commodity,rev_side_label,rev_source,n_episodes,n_ended,n_censored,volume_divergence_rate,mean_active_duration_obs,median_active_duration_obs,q25_active_duration_obs,q75_active_duration_obs,mean_duration_to_end_signal_obs,median_duration_to_end_signal_obs,duration_unit,duration_definition
0,AG,DOWN,TSREV,104,104,0,0.605769,2.913462,3.0,1.00,4.00,2.913462,3.0,validated_active_observations,Duration / remaining-life metrics count valida...
1,AG,UP,TSREV,80,80,0,0.675000,2.887500,3.0,1.00,4.00,2.887500,3.0,validated_active_observations,Duration / remaining-life metrics count valida...
2,CF,DOWN,FastREV,9,9,0,0.444444,6.111111,6.0,3.00,9.00,6.111111,6.0,validated_active_observations,Duration / remaining-life metrics count valida...
3,CF,UP,FastREV,11,11,0,0.454545,2.181818,2.0,1.50,2.50,2.181818,2.0,validated_active_observations,Duration / remaining-life metrics count valida...
4,FG,DOWN,FastREV,16,16,0,0.562500,1.687500,1.5,1.00,2.00,1.687500,1.5,validated_active_observations,Duration / remaining-life metrics count valida...
5,FG,UP,FastREV,16,16,0,0.562500,1.312500,1.0,1.00,2.00,1.312500,1.0,validated_active_observations,Duration / remaining-life metrics count valida...
6,M,DOWN,TSREV,32,32,0,0.656250,10.656250,6.0,2.75,13.75,10.656250,6.0,validated_active_observations,Duration / remaining-life metrics count valida...
7,M,UP,TSREV,27,27,0,0.777778,12.259259,6.0,2.00,18.50,12.259259,6.0,validated_active_observations,Duration / remaining-life metrics count valida...
8,MA,DOWN,FastREV,8,8,0,0.375000,2.750000,2.5,1.00,4.00,2.750000,2.5,validated_active_observations,Duration / remaining-life metrics count valida...
9,MA,UP,FastREV,11,11,0,0.909091,4.363636,3.0,2.00,6.00,4.363636,3.0,validated_active_observations,Duration / remaining-life metrics count valida...


,commodity,rev_side_label,rev_source,horizon_obs,n_state_obs,n_end_within_horizon_obs,p_end_within_horizon_obs,p_survive_beyond_horizon_obs,duration_unit,duration_definition
0,AG,DOWN,TSREV,1,303,104,0.343234,0.656766,validated_active_observations,Duration / remaining-life metrics count valida...
1,AG,DOWN,TSREV,2,303,174,0.574257,0.425743,validated_active_observations,Duration / remaining-life metrics count valida...
2,AG,DOWN,TSREV,3,303,227,0.749175,0.250825,validated_active_observations,Duration / remaining-life metrics count valida...
3,AG,DOWN,TSREV,5,303,286,0.943894,0.056106,validated_active_observations,Duration / remaining-life metrics count valida...
4,AG,DOWN,TSREV,10,303,302,0.996700,0.003300,validated_active_observations,Duration / remaining-life metrics count valida...
5,AG,DOWN,TSREV,20,303,303,1.000000,0.000000,validated_active_observations,Duration / remaining-life metrics count valida...
6,AG,UP,TSREV,1,231,80,0.346320,0.653680,validated_active_observations,Duration / remaining-life metrics count valida...
7,AG,UP,TSREV,2,231,135,0.584416,0.415584,validated_active_observations,Duration / remaining-life metrics count valida...
8,AG,UP,TSREV,3,231,176,0.761905,0.238095,validated_active_observations,Duration / remaining-life metrics count valida...
9,AG,UP,TSREV,5,231,218,0.943723,0.056277,validated_active_observations,Duration / remaining-life metrics count valida...


,commodity,date,reversal_active_now,rev_side_label,rev_source,rev_validated_age_obs,reversal_end_confirmed,n_comparable_completed,duration_reliability,duration_unit,duration_definition,mean_remaining_obs,median_remaining_obs,p_end_within_1obs,p_end_within_2obs,p_end_within_3obs,p_end_within_5obs,p_end_within_10obs,p_end_within_20obs
0,AG,2026-06-30,False,NONE,NONE,<NA>,True,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,CF,2026-06-30,False,NONE,NONE,<NA>,False,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,EB,2026-06-30,False,NONE,NONE,<NA>,False,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FG,2026-06-30,False,NONE,NONE,<NA>,False,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,LH,2026-06-30,False,NONE,NONE,<NA>,False,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,M,2026-06-30,False,NONE,NONE,<NA>,False,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,MA,2026-06-30,False,NONE,NONE,<NA>,False,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,SA,2026-06-30,True,UP,RAREV,16,False,10,OK,validated_active_observations,Duration / remaining-life metrics count valida...,45.300000,53.0,0.000000,0.000000,0.0,0.0,0.0,0.3
8,SP,2026-06-30,False,NONE,NONE,<NA>,False,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,SR,2026-06-30,True,DOWN,TSREV,1,False,107,OK,validated_active_observations,Duration / remaining-life metrics count valida...,1.308411,1.0,0.766355,0.925234,1.0,1.0,1.0,1.0


In [22]:
REQUIRED_REV_FINAL_OBJECTS = [
    "panel_state_duration",
    "rev_best",
    "reversal_duration_summary",
    "reversal_survival_table",
    "current_reversal_life_report_df",
]
missing = [x for x in REQUIRED_REV_FINAL_OBJECTS if x not in globals()]
if missing:
    raise RuntimeError(f"Missing required objects. Run previous cells first: {missing}")


def _select(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    return df[[c for c in cols if c in df.columns]].copy()


def _p_end_key(col: str) -> int:
    # Supports both the new obs suffix and older d suffix.
    # Examples: p_end_within_5obs -> 5; p_end_within_5d -> 5.
    suffix = col.replace("p_end_within_", "")
    digits = "".join(ch for ch in suffix if ch.isdigit())
    return int(digits) if digits else 10**9


def _reliability(n) -> str:
    if pd.isna(n) or n < 5:
        return "INSUFFICIENT"
    if n < 20:
        return "LOW"
    if n < 50:
        return "MEDIUM"
    return "HIGH"


def _rev_status(row: pd.Series) -> str:
    end_flag = bool(row.get("latest_row_has_reversal_end_flag", row.get("reversal_end_confirmed", False)))
    if end_flag:
        return "REVERSAL_CONFIRMED_ENDED"
    if not bool(row.get("reversal_active_now", False)):
        return "NO_VALIDATED_REVERSAL"

    side = row.get("rev_side_label", "NONE")
    source = row.get("rev_source", "NONE")
    return f"{side}_REVERSAL_ACTIVE_{source}"


def _standardize_reversal_age_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Final-output naming rule:
    use only rev_validated_age_obs for the age of the current validated reversal episode.

    Older aliases are normalized here:
    rev_validated_age / REVValidatedAge / ReversalAge / REVAge / rev_age.
    """
    out = df.copy()
    primary = "rev_validated_age_obs"

    age_aliases = [
        "rev_validated_age_obs",
        "rev_validated_age",
        "REVValidatedAge",
        "ReversalAge",
        "REVAge",
        "rev_age",
        "rev_validated_age_obs_x",
        "rev_validated_age_obs_y",
        "rev_validated_age_x",
        "rev_validated_age_y",
    ]

    existing = [c for c in age_aliases if c in out.columns]
    if existing:
        age = pd.Series(np.nan, index=out.index, dtype="float64")
        for c in existing:
            age = age.combine_first(pd.to_numeric(out[c], errors="coerce"))

        out = out.drop(columns=[c for c in existing if c != primary], errors="ignore")
        out[primary] = age

    return out


latest = (
    panel_state_duration
    .sort_values(["commodity", "date"])
    .groupby("commodity", sort=False)
    .tail(1)
    .copy()
)

p_cols = sorted(
    [c for c in current_reversal_life_report_df.columns if c.startswith("p_end_within_")],
    key=_p_end_key,
)

life = _select(current_reversal_life_report_df, [
    "commodity",
    "reversal_active_now",
    "rev_side_label",
    "rev_source",
    "rev_validated_age_obs",
    "reversal_end_confirmed",
    "n_comparable_completed",
    "duration_reliability",
    "duration_unit",
    "duration_definition",
    "mean_remaining_obs",
    "median_remaining_obs",
] + p_cols)

# Keep the older field for compatibility, but expose a clearer final-output alias.
if "reversal_end_confirmed" in life.columns:
    life["latest_row_has_reversal_end_flag"] = life["reversal_end_confirmed"].fillna(False).astype(bool)
else:
    life["latest_row_has_reversal_end_flag"] = False

best_cols = [
    "commodity",
    "best_model_label",
    "best_rev_type",
    "best_J",
    "best_L",
    "best_K",
    "best_n",
    "best_mean_y",
    "best_t_nw",
    "best_hit_rate",
    "best_boot_p_mean_le_0",
    "best_mean_raw",
    "best_candidate_model",
    "best_confirmed_model",
    "best_model_quality",
]

if rev_best.empty:
    best = pd.DataFrame(columns=best_cols)
else:
    best = _select(rev_best, [
        "commodity",
        "label",
        "rev_type",
        "J",
        "L",
        "K",
        "n",
        "mean_y",
        "t_nw",
        "hit_rate",
        "boot_p_mean_le_0",
        "mean_raw",
        "candidate_model",
        "confirmed_model",
        "model_quality",
    ]).rename(columns={
        "label": "best_model_label",
        "rev_type": "best_rev_type",
        "J": "best_J",
        "L": "best_L",
        "K": "best_K",
        "n": "best_n",
        "mean_y": "best_mean_y",
        "t_nw": "best_t_nw",
        "hit_rate": "best_hit_rate",
        "boot_p_mean_le_0": "best_boot_p_mean_le_0",
        "mean_raw": "best_mean_raw",
        "candidate_model": "best_candidate_model",
        "confirmed_model": "best_confirmed_model",
        "model_quality": "best_model_quality",
    })

    for col in best_cols:
        if col not in best.columns:
            best[col] = np.nan

    best = best[best_cols]

duration = reversal_duration_summary.copy().rename(columns={
    "n_episodes": "hist_n_episodes",
    "n_ended": "hist_n_ended",
    "n_censored": "hist_n_censored",
    "volume_divergence_rate": "hist_volume_divergence_rate",

    # New obs-based Cell 14 fields.
    "mean_active_duration_obs": "hist_mean_active_duration_obs",
    "median_active_duration_obs": "hist_median_active_duration_obs",
    "q25_active_duration_obs": "hist_q25_active_duration_obs",
    "q75_active_duration_obs": "hist_q75_active_duration_obs",
    "mean_duration_to_end_signal_obs": "hist_mean_duration_to_end_signal_obs",
    "median_duration_to_end_signal_obs": "hist_median_duration_to_end_signal_obs",

    # Backward compatibility with older Cell 14 outputs.
    "mean_active_duration": "hist_mean_active_duration_obs",
    "median_active_duration": "hist_median_active_duration_obs",
    "q25_active_duration": "hist_q25_active_duration_obs",
    "q75_active_duration": "hist_q75_active_duration_obs",
    "mean_duration_to_end_signal": "hist_mean_duration_to_end_signal_obs",
    "median_duration_to_end_signal": "hist_median_duration_to_end_signal_obs",
})

if not duration.empty and "hist_n_ended" in duration.columns:
    duration["hist_duration_reliability"] = duration["hist_n_ended"].apply(_reliability)

q1_current = _select(latest, [
    "commodity",
    "commodity_name",
    "main_symbol",
    "date",
    "close",
    "rev_best_type",
    "J_rev_star",
    "L_rev_star",
    "K_rev_eval",
    "rev_model_quality",
    "rev_model_scope",
    "REVModelUsable",
    "REVSource",
    "REVSide",
    "REVSignal",
    "REVValidatedSignal",
    "REVVolumeDivergence",
    "REVEpisodeID",
    "REVEpisodeSideLabel",
    "REVEpisodeSource",
    "REVValidatedAge",
    "REVEpisodeStartDate",
]).merge(best, on="commodity", how="left")

q2_current = _select(latest, [
    "commodity",
    "commodity_name",
    "main_symbol",
    "date",
    "close",
    "rev_best_type",
    "J_rev_star",
    "L_rev_star",
    "K_rev_eval",
    "rev_model_quality",
    "rev_model_scope",
    "REVModelUsable",
    "REVEpisodeID",
    "REVEpisodeSideLabel",
    "REVEpisodeSource",
    "REVValidatedAge",
    "REVEpisodeStartDate",
    "REVEpisodeCensored",
]).merge(life, on="commodity", how="left")

if not duration.empty:
    q2_current["rev_side_for_duration"] = q2_current["rev_side_label"].where(
        q2_current["reversal_active_now"].fillna(False)
    )
    q2_current["rev_source_for_duration"] = q2_current["rev_source"].where(
        q2_current["reversal_active_now"].fillna(False)
    )

    duration_for_current = duration.drop(
        columns=["duration_unit", "duration_definition"],
        errors="ignore",
    )

    q2_current = q2_current.merge(
        duration_for_current.rename(columns={
            "rev_side_label": "rev_side_for_duration",
            "rev_source": "rev_source_for_duration",
        }),
        on=["commodity", "rev_side_for_duration", "rev_source_for_duration"],
        how="left",
    ).drop(columns=["rev_side_for_duration", "rev_source_for_duration"])

q2_current["life_estimate_reliability"] = (
    q2_current["duration_reliability"]
    if "duration_reliability" in q2_current.columns
    else "UNKNOWN"
)

q3_current = _select(latest, [
    "commodity",
    "commodity_name",
    "main_symbol",
    "date",
    "close",
    "REVSource",
    "REVSide",
    "REVSignal",
    "REVValidatedSignal",
    "REVValidatedAge",
    "ReversalEnd_signal",
    "ReversalEnd_flip",
    "ReversalEnd_fast_absorbed",
    "ReversalEnd",
    "REVEpisodeID",
    "REVEpisodeEndDate",
    "REVEpisodeEndReason",
]).merge(
    _select(life, [
        "commodity",
        "reversal_end_confirmed",
        "latest_row_has_reversal_end_flag",
    ] + p_cols),
    on="commodity",
    how="left",
)

q1_current = _standardize_reversal_age_columns(q1_current)
q2_current = _standardize_reversal_age_columns(q2_current)
q3_current = _standardize_reversal_age_columns(q3_current)

core = (
    q1_current
    .merge(_select(q2_current, [
        "commodity",
        "reversal_active_now",
        "rev_side_label",
        "rev_source",
        "rev_validated_age_obs",
        "reversal_end_confirmed",
        "latest_row_has_reversal_end_flag",
        "n_comparable_completed",
        "duration_reliability",
        "duration_unit",
        "duration_definition",
        "mean_remaining_obs",
        "median_remaining_obs",
        "life_estimate_reliability",
    ] + p_cols), on="commodity", how="left")
    .merge(_select(q3_current, [
        "commodity",
        "ReversalEnd_signal",
        "ReversalEnd_flip",
        "ReversalEnd_fast_absorbed",
        "ReversalEnd",
    ]), on="commodity", how="left", suffixes=("", "_end"))
)

core = _standardize_reversal_age_columns(core)
core["reversal_status"] = core.apply(_rev_status, axis=1)

front = [
    "commodity",
    "commodity_name",
    "main_symbol",
    "date",
    "close",
    "reversal_status",
    "life_estimate_reliability",
]
core = core[[c for c in front if c in core.columns] + [c for c in core.columns if c not in front]]

_state = panel_state_duration.copy()
_state["rev_signal_flag"] = _state["REVSignal"].fillna(0).astype(bool)
_state["rev_validated_flag"] = _state["REVValidatedSignal"].fillna(0).astype(bool)
_state["rev_up_flag"] = _state["rev_validated_flag"] & (_state["REVSide"] > 0)
_state["rev_down_flag"] = _state["rev_validated_flag"] & (_state["REVSide"] < 0)

q1_history = (
    _select(_state, ["commodity", "commodity_name", "main_symbol"])
    .drop_duplicates("commodity")
    .merge(_state.groupby("commodity").agg(
        first_date=("date", "min"),
        last_date=("date", "max"),
        total_obs=("date", "size"),
        rev_signal_obs=("rev_signal_flag", "sum"),
        rev_validated_obs=("rev_validated_flag", "sum"),
        up_validated_obs=("rev_up_flag", "sum"),
        down_validated_obs=("rev_down_flag", "sum"),
    ).reset_index(), on="commodity", how="left")
    .merge(best, on="commodity", how="left")
)

q1_history["rev_signal_rate"] = q1_history["rev_signal_obs"] / q1_history["total_obs"]
q1_history["rev_validated_rate"] = q1_history["rev_validated_obs"] / q1_history["total_obs"]
q1_history["up_share_when_validated"] = (
    q1_history["up_validated_obs"] / q1_history["rev_validated_obs"].replace(0, np.nan)
)
q1_history["down_share_when_validated"] = (
    q1_history["down_validated_obs"] / q1_history["rev_validated_obs"].replace(0, np.nan)
)

q2_history = duration.copy()

q3_history = reversal_survival_table.copy()
if {"commodity", "rev_side_label", "rev_source"}.issubset(q3_history.columns) and not duration.empty:
    q3_history = q3_history.merge(
        _select(duration, [
            "commodity",
            "rev_side_label",
            "rev_source",
            "hist_n_episodes",
            "hist_n_ended",
            "hist_volume_divergence_rate",
            "hist_duration_reliability",
        ]),
        on=["commodity", "rev_side_label", "rev_source"],
        how="left",
    )

if "n_state_obs" in q3_history.columns:
    q3_history["survival_sample_reliability"] = q3_history["n_state_obs"].apply(_reliability)

# Keep final age naming consistent across all current-output CSVs.
for _df_name in ["q1_current", "q2_current", "q3_current", "core"]:
    _df = globals()[_df_name]
    _df = _standardize_reversal_age_columns(_df)

    if "rev_validated_age_obs" in _df.columns:
        front_cols = [c for c in _df.columns if c != "rev_validated_age_obs"]
        insert_at = min(len(front_cols), 12)
        _df = _df[front_cols[:insert_at] + ["rev_validated_age_obs"] + front_cols[insert_at:]]

    globals()[_df_name] = _df


outputs = {
    "reversal_core_current_summary.csv": core,
    "reversal_q1_existence_current.csv": q1_current,
    "reversal_q2_duration_current.csv": q2_current,
    "reversal_q3_end_current.csv": q3_current,
    "reversal_hist_q1_existence_model.csv": q1_history,
    "reversal_hist_q2_duration_distribution.csv": q2_history,
    "reversal_hist_q3_end_survival.csv": q3_history,
}

for filename, df_out in outputs.items():
    df_out.to_csv(RESULT_DIR / filename, index=False)

manifest = pd.DataFrame([
    {
        "file": "reversal_core_current_summary.csv",
        "scope": "CURRENT",
        "question": "ALL",
        "description": (
            "当前反转主表：反转存在、持续周期、结束状态、最佳反转模型汇总；"
            "duration/remaining-life 单位为 validated-active observations"
        ),
    },
    {
        "file": "reversal_q1_existence_current.csv",
        "scope": "CURRENT",
        "question": "Q1",
        "description": "当前是否存在明确反转：方向、来源、模型有效性、validated reversal age obs",
    },
    {
        "file": "reversal_q2_duration_current.csv",
        "scope": "CURRENT",
        "question": "Q2",
        "description": (
            "当前反转还能持续多久：剩余寿命、未来 N 个 validated-active observations 内结束概率、"
            "历史同方向 duration 摘要"
        ),
    },
    {
        "file": "reversal_q3_end_current.csv",
        "scope": "CURRENT",
        "question": "Q3",
        "description": "当前反转是否结束：信号消失、方向翻转、快速反转吸收；latest_row_has_reversal_end_flag 表示最新行是否有结束标记",
    },
    {
        "file": "reversal_hist_q1_existence_model.csv",
        "scope": "FULL_HISTORY",
        "question": "Q1",
        "description": "全历史反转存在频率、方向分布、最佳反转模型有效性；计数单位为 observations",
    },
    {
        "file": "reversal_hist_q2_duration_distribution.csv",
        "scope": "FULL_HISTORY",
        "question": "Q2",
        "description": "全历史反转 episode 持续时间分布：均值、中位数、分位数、样本数；duration 单位为 validated-active observations",
    },
    {
        "file": "reversal_hist_q3_end_survival.csv",
        "scope": "FULL_HISTORY",
        "question": "Q3",
        "description": "全历史反转结束概率：按方向、来源、horizon_obs 统计未来结束概率",
    },
])
manifest.to_csv(RESULT_DIR / "reversal_output_manifest.csv", index=False)

display(core)
display(manifest)
print("Saved current + full-history reversal CSVs to:", RESULT_DIR.resolve())

,commodity,commodity_name,main_symbol,date,close,reversal_status,life_estimate_reliability,rev_best_type,J_rev_star,L_rev_star,K_rev_eval,rev_model_quality,rev_validated_age_obs,rev_model_scope,REVModelUsable,REVSource,REVSide,REVSignal,REVValidatedSignal,REVVolumeDivergence,REVEpisodeID,REVEpisodeSideLabel,REVEpisodeSource,REVEpisodeStartDate,best_model_label,best_rev_type,best_J,best_L,best_K,best_n,best_mean_y,best_t_nw,best_hit_rate,best_boot_p_mean_le_0,best_mean_raw,best_candidate_model,best_confirmed_model,best_model_quality,reversal_active_now,rev_side_label,rev_source,reversal_end_confirmed,latest_row_has_reversal_end_flag,n_comparable_completed,duration_reliability,duration_unit,duration_definition,mean_remaining_obs,median_remaining_obs,p_end_within_1obs,p_end_within_2obs,p_end_within_3obs,p_end_within_5obs,p_end_within_10obs,p_end_within_20obs,ReversalEnd_signal,ReversalEnd_flip,ReversalEnd_fast_absorbed,ReversalEnd
0,AG,沪银,KQ.m@SHFE.ag,2026-06-30,13966.0,REVERSAL_CONFIRMED_ENDED,NO_ACTIVE_REVERSAL,TSREV,5.0,0.0,5.0,CANDIDATE,NaN,CANDIDATE,True,NONE,NaN,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,AG_TSREV_J5_K5,TSREV,5,0,5,828,0.071643,1.066113,0.543478,0.1425,0.002856,True,False,CANDIDATE,False,NONE,NONE,True,True,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,0.0,0.0,1.0
1,CF,棉花,KQ.m@CZCE.CF,2026-06-30,16065.0,NO_VALIDATED_REVERSAL,NO_ACTIVE_REVERSAL,FastREV,60.0,0.0,20.0,CANDIDATE,NaN,CANDIDATE,True,NONE,NaN,0.0,0.0,1.0,<NA>,<NA>,<NA>,<NA>,CF_FastREV_J60_K20,FastREV,60,0,20,109,0.579930,3.729932,0.678899,0.0000,0.011658,True,False,CANDIDATE,False,NONE,NONE,False,False,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
2,EB,苯乙烯,KQ.m@DCE.eb,2026-06-30,7368.0,NO_VALIDATED_REVERSAL,NO_ACTIVE_REVERSAL,FastREV,60.0,0.0,20.0,DIAGNOSTIC_ONLY,NaN,DIAGNOSTIC_ONLY,False,FastREV,1.0,1.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,EB_FastREV_J60_K20,FastREV,60,0,20,57,0.606555,4.244212,0.789474,0.0000,0.049736,False,False,DIAGNOSTIC_ONLY,False,NONE,NONE,False,False,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
3,FG,玻璃,KQ.m@CZCE.FG,2026-06-30,967.0,NO_VALIDATED_REVERSAL,NO_ACTIVE_REVERSAL,FastREV,3.0,0.0,10.0,CANDIDATE,NaN,CANDIDATE,True,NONE,NaN,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,FG_FastREV_J3_K10,FastREV,3,0,10,125,0.217974,1.611736,0.608000,0.0750,0.011357,True,False,CANDIDATE,False,NONE,NONE,False,False,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
4,LH,生猪,KQ.m@DCE.lh,2026-06-30,12365.0,NO_VALIDATED_REVERSAL,NO_ACTIVE_REVERSAL,FastREV,20.0,0.0,20.0,DIAGNOSTIC_ONLY,NaN,DIAGNOSTIC_ONLY,False,NONE,NaN,0.0,0.0,1.0,<NA>,<NA>,<NA>,<NA>,LH_FastREV_J20_K20,FastREV,20,0,20,103,0.219180,0.414436,0.533981,0.5475,0.016174,False,False,DIAGNOSTIC_ONLY,False,NONE,NONE,False,False,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
5,M,豆粕,KQ.m@DCE.m,2026-06-30,2951.0,NO_VALIDATED_REVERSAL,NO_ACTIVE_REVERSAL,TSREV,60.0,0.0,20.0,CONFIRMED,NaN,CONFIRMED,True,NONE,NaN,0.0,0.0,1.0,<NA>,<NA>,<NA>,<NA>,M_TSREV_J60_K20,TSREV,60,0,20,762,0.258449,2.112081,0.578740,0.0100,0.011440,True,True,CONFIRMED,False,NONE,NONE,False,False,0,NO_ACTIVE_REVERSAL,validated_active_observations,Duration / remaining-life metrics count valida...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0
6,MA,甲醇,KQ.m@CZCE.MA,2026-06-30,2442.0,NO_VALIDATED_REVERSAL,NO_ACTIVE_REVERSAL,FastREV,60.0,0.0,10.0,CANDIDATE,NaN,CANDIDATE,True,NONE,NaN,0.0,0.0,0.0,<NA>,<NA>,<NA>,<NA>,MA_FastREV_J60_K10,FastREV,60,0,10,96,0.797533,5.791973,0.781250,0.0000,0.037858,True,False,CANDIDATE,False,NONE,NONE,False,False,0,NO_ACTIVE_REVERSAL,valid

,file,scope,question,description
0,reversal_core_current_summary.csv,CURRENT,ALL,当前反转主表：反转存在、持续周期、结束状态、最佳反转模型汇总；duration/remain...
1,reversal_q1_existence_current.csv,CURRENT,Q1,当前是否存在明确反转：方向、来源、模型有效性、validated reversal age obs
2,reversal_q2_duration_current.csv,CURRENT,Q2,当前反转还能持续多久：剩余寿命、未来 N 个 validated-active observ...
3,reversal_q3_end_current.csv,CURRENT,Q3,当前反转是否结束：信号消失、方向翻转、快速反转吸收；latest_row_has_rever...
4,reversal_hist_q1_existence_model.csv,FULL_HISTORY,Q1,全历史反转存在频率、方向分布、最佳反转模型有效性；计数单位为 observations
5,reversal_hist_q2_duration_distribution.csv,FULL_HISTORY,Q2,全历史反转 episode 持续时间分布：均值、中位数、分位数、样本数；duration 单...
6,reversal_hist_q3_end_survival.csv,FULL_HISTORY,Q3,全历史反转结束概率：按方向、来源、horizon_obs 统计未来结束概率


Saved current + full-history reversal CSVs to: /home/zilinm2/main_continuous_daily_trend_momentum_reversal_research/results
